<a href="https://colab.research.google.com/github/shreyatembare54-sys/VoltRelay_Energy_Data_Analytics_Hackathon/blob/main/VoltRelay_Energy_Data_Analytics_Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

print("Files available in Colab:")
print("=" * 50)

for file in sorted(os.listdir("/content")):
    print(file)


Files available in Colab:
.config
batteries.csv
city_daily_context.csv
drive
fleet_partners.csv
riders.csv
sample_data
station_hourly_status.csv
stations.csv
support_tickets.csv
swap_events.csv


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
from pathlib import Path

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid")

print("✅ All libraries loaded successfully!")

✅ All libraries loaded successfully!


In [ ]:
from pathlib import Path

data_path = Path("/content")

print("📁 Dataset sizes")
print("=" * 60)

for file in sorted(data_path.glob("*.csv")):
    size_mb = file.stat().st_size / (1024 * 1024)
    print(f"{file.name:<35} {size_mb:>10.2f} MB")

📁 Dataset sizes
batteries.csv                             0.60 MB
city_daily_context.csv                    0.21 MB
fleet_partners.csv                        0.00 MB
riders.csv                                2.00 MB
station_hourly_status.csv               103.33 MB
stations.csv                              0.02 MB
support_tickets.csv                       6.23 MB
swap_events.csv                         694.97 MB


In [ ]:
# STEP 4: Load supporting datasets

batteries = pd.read_csv("/content/batteries.csv")
city_context = pd.read_csv("/content/city_daily_context.csv")
fleet_partners = pd.read_csv("/content/fleet_partners.csv")
riders = pd.read_csv("/content/riders.csv")
station_hourly = pd.read_csv("/content/station_hourly_status.csv")
stations = pd.read_csv("/content/stations.csv")
support_tickets = pd.read_csv("/content/support_tickets.csv")

print("✅ Supporting datasets loaded!")
print()

print("Dataset shapes:")
print("-" * 50)
print("Batteries          :", batteries.shape)
print("City Context       :", city_context.shape)
print("Fleet Partners     :", fleet_partners.shape)
print("Riders             :", riders.shape)
print("Station Hourly     :", station_hourly.shape)
print("Stations           :", stations.shape)
print("Support Tickets    :", support_tickets.shape)

✅ Supporting datasets loaded!

Dataset shapes:
--------------------------------------------------
Batteries          : (6500, 13)
City Context       : (3282, 12)
Fleet Partners     : (12, 12)
Riders             : (20000, 12)
Station Hourly     : (1487712, 14)
Stations           : (152, 22)
Support Tickets    : (44000, 12)


In [ ]:
# STEP 5: Load the main swap events dataset

print("⏳ Loading swap_events.csv...")
print("This may take a little time because the file is large.")

swap_events = pd.read_csv(
    "/content/swap_events.csv",
    low_memory=False
)

print("\n✅ swap_events loaded successfully!")
print("Shape:", swap_events.shape)

print("\nColumns:")
print(swap_events.columns.tolist())

⏳ Loading swap_events.csv...
This may take a little time because the file is large.

✅ swap_events loaded successfully!
Shape: (3877013, 22)

Columns:
['event_id', 'rider_id', 'station_id', 'event_ts', 'event_type', 'attempt_seq', 'queue_wait_sec', 'battery_in_id', 'battery_out_id', 'soc_in_pct', 'soh_in_pct', 'soc_out_pct', 'soh_out_pct', 'km_since_last_swap', 'tariff_code', 'list_price_inr', 'discount_inr', 'amount_charged_inr', 'payment_mode', 'energy_to_recharge_kwh', 'station_firmware', 'sync_mode']


In [ ]:
# STEP 6: Inspect swap events

print("📊 First 5 rows:")
display(swap_events.head())

print("\n📋 Data types:")
display(swap_events.dtypes)

print("\n❓ Missing values:")
missing = swap_events.isnull().sum().sort_values(ascending=False)
display(missing[missing > 0])

print("\n🔢 Event types:")
display(swap_events["event_type"].value_counts(dropna=False))


📊 First 5 rows:


,event_id,rider_id,station_id,event_ts,event_type,attempt_seq,queue_wait_sec,battery_in_id,battery_out_id,soc_in_pct,soh_in_pct,soc_out_pct,soh_out_pct,km_since_last_swap,tariff_code,list_price_inr,discount_inr,amount_charged_inr,payment_mode,energy_to_recharge_kwh,station_firmware,sync_mode
0,SW-000000001,RD-1000003,STN-MUM-115,2024-01-01 11:10:56,swap_completed,1,88,BAT-2W-103542,BAT-2W-102547,15.00,100.60,98.70,98.90,80.40,PARTNER,60.00,6.00,54.00,partner_invoice,2.04,v3.2.1,realtime
1,SW-000000002,RD-1000004,STN-HYD-070,2024-01-01 17:13:06,swap_completed,1,148,BAT-2W-105296,BAT-2W-101923,8.30,98.30,97.30,98.20,66.10,PARTNER,60.00,9.00,51.00,partner_invoice,2.16,v3.2.0,realtime
2,SW-000000003,RD-1000005,STN-DEL-046,2024-01-01 20:15:14,swap_completed,1,307,BAT-2W-104732,BAT-2W-102492,11.50,99.10,99.80,99.10,66.60,STD,60.00,0.00,60.00,partner_invoice,2.04,v3.2.0,realtime
3,SW-000000004,RD-1000006,STN-JAI-135,2024-01-01 20:26:07,swap_completed,1,399,BAT-2W-103839,BAT-2W-103267,7.90,99.60,98.40,98.20,65.40,STD,60.00,0.00,60.00,partner_invoice,2.19,v3.2.0,realtime
4,SW-000000005,RD-1000012,STN-PUN-100,2024-01-01 21:13:07,swap_completed,1,201,BAT-2W-104056,BAT-2W-106218,10.50,99.20,95.60,99.90,73.80,STD,60.00,0.00,60.00,partner_invoice,1.97,v3.2.1,realtime



📋 Data types:


,0
event_id,object
rider_id,object
station_id,object
event_ts,object
event_type,object
attempt_seq,int64
queue_wait_sec,int64
battery_in_id,object
battery_out_id,object
soc_in_pct,float64



❓ Missing values:


,0
soc_out_pct,231204
battery_out_id,231204
energy_to_recharge_kwh,231204
soh_out_pct,231204
soc_in_pct,28282
soh_in_pct,28282
km_since_last_swap,28282
battery_in_id,28282



🔢 Event types:


,count
event_type,
swap_completed,3645809
failed_no_charged_battery,134389
abandoned_queue,68533
cancelled_by_rider,16712
failed_system_error,11570


In [ ]:
# STEP 7: Initial data-quality audit

print("🔍 DATA QUALITY AUDIT")
print("=" * 60)

# 1. Duplicate rows
print("\n1️⃣ Exact duplicate rows:")
print(swap_events.duplicated().sum())

# 2. Unique event types
print("\n2️⃣ Event types:")
print(swap_events["event_type"].value_counts(dropna=False))

# 3. Payment modes
print("\n3️⃣ Payment modes:")
print(swap_events["payment_mode"].value_counts(dropna=False))

# 4. Tariff codes
print("\n4️⃣ Tariff codes:")
print(swap_events["tariff_code"].value_counts(dropna=False))

# 5. Station firmware
print("\n5️⃣ Station firmware:")
print(swap_events["station_firmware"].value_counts(dropna=False))

# 6. Sync modes
print("\n6️⃣ Sync modes:")
print(swap_events["sync_mode"].value_counts(dropna=False))

# 7. Date range
print("\n7️⃣ Event timestamp range:")
print("Minimum:", swap_events["event_ts"].min())
print("Maximum:", swap_events["event_ts"].max())

# 8. Basic numeric summary
print("\n8️⃣ Key numeric variables:")
display(
    swap_events[
        [
            "queue_wait_sec",
            "soc_in_pct",
            "soh_in_pct",
            "soc_out_pct",
            "soh_out_pct",
            "km_since_last_swap",
            "list_price_inr",
            "discount_inr",
            "amount_charged_inr",
            "energy_to_recharge_kwh"
        ]
    ].describe().T
)

🔍 DATA QUALITY AUDIT

1️⃣ Exact duplicate rows:
0

2️⃣ Event types:
event_type
swap_completed               3645809
failed_no_charged_battery     134389
abandoned_queue                68533
cancelled_by_rider             16712
failed_system_error            11570
Name: count, dtype: int64

3️⃣ Payment modes:
payment_mode
partner_invoice    3877013
Name: count, dtype: int64

4️⃣ Tariff codes:
tariff_code
PARTNER    2565478
STD        1095936
PEAK        177879
OFFPEAK      37720
Name: count, dtype: int64

5️⃣ Station firmware:
station_firmware
v3.2.1    2315839
v3.2.0    1561174
Name: count, dtype: int64

6️⃣ Sync modes:
sync_mode
realtime         3285899
offline_batch     591114
Name: count, dtype: int64

7️⃣ Event timestamp range:
Minimum: 2024-01-01 00:00:33
Maximum: 2025-06-30 23:59:23

8️⃣ Key numeric variables:


,count,mean,std,min,25%,50%,75%,max
queue_wait_sec,"3,877,013.00",242.95,163.91,0.00,137.00,206.00,305.00,"2,400.00"
soc_in_pct,"3,848,731.00",9.29,4.45,1.00,6.00,9.30,12.50,28.40
soh_in_pct,"3,848,731.00",88.17,7.97,55.20,84.00,89.30,94.10,101.40
soc_out_pct,"3,645,809.00",96.66,3.37,80.00,95.90,97.30,98.60,103.00
soh_out_pct,"3,645,809.00",88.41,7.45,54.70,84.30,89.20,93.90,101.30
km_since_last_swap,"3,848,731.00",62.04,20.44,-20.00,51.90,60.00,69.00,420.00
list_price_inr,"3,877,013.00",68.81,13.88,60.00,65.00,65.00,65.00,110.00
discount_inr,"3,877,013.00",5.84,5.64,0.00,0.00,6.50,7.80,30.80
amount_charged_inr,"3,877,013.00",60.13,20.70,0.00,55.20,58.50,65.00,135.00
energy_to_recharge_kwh,"3,645,809.00",2.09,0.81,0.81,1.72,1.87,2.03,5.52


In [ ]:
# STEP 8: Timestamp preparation and firmware timestamp audit

# Convert event timestamp to datetime
swap_events["event_ts"] = pd.to_datetime(
    swap_events["event_ts"],
    errors="coerce"
)

# Check invalid timestamps
invalid_ts = swap_events["event_ts"].isna().sum()

print("🕒 TIMESTAMP AUDIT")
print("=" * 60)
print("Invalid timestamps:", invalid_ts)
print("Earliest timestamp :", swap_events["event_ts"].min())
print("Latest timestamp   :", swap_events["event_ts"].max())

# Known firmware issue window
affected = swap_events[
    (swap_events["station_firmware"] == "v3.2.0") &
    (swap_events["event_ts"] >= "2025-03-10") &
    (swap_events["event_ts"] <= "2025-04-14 23:59:59")
]

print("\n⚠️ Potential v3.2.0 affected records:", len(affected))

if len(affected) > 0:
    print("\nAffected records by date:")
    display(
        affected["event_ts"]
        .dt.date
        .value_counts()
        .sort_index()
        .head(20)
    )

    print("\nAffected records by firmware:")
    display(affected["station_firmware"].value_counts())


🕒 TIMESTAMP AUDIT
Invalid timestamps: 0
Earliest timestamp : 2024-01-01 00:00:33
Latest timestamp   : 2025-06-30 23:59:23

⚠️ Potential v3.2.0 affected records: 139490

Affected records by date:


,count
event_ts,
2025-03-10,3609
2025-03-11,3708
2025-03-12,3643
2025-03-13,3670
2025-03-14,2510
2025-03-15,3951
2025-03-16,3916
2025-03-17,3787
2025-03-18,3730



Affected records by firmware:


,count
station_firmware,
v3.2.0,139490


In [ ]:
# STEP 9: Firmware and timestamp coverage

print("🔧 FIRMWARE DISTRIBUTION")
print("=" * 60)

firmware_counts = (
    swap_events["station_firmware"]
    .value_counts(dropna=False)
    .rename_axis("station_firmware")
    .reset_index(name="event_count")
)

firmware_counts["percentage"] = (
    firmware_counts["event_count"]
    / len(swap_events) * 100
)

display(firmware_counts)

print("\n📅 EVENTS BY MONTH")
print("=" * 60)

monthly_events = (
    swap_events
    .assign(month=swap_events["event_ts"].dt.to_period("M").astype(str))
    .groupby("month")
    .size()
    .reset_index(name="event_count")
)

display(monthly_events)


🔧 FIRMWARE DISTRIBUTION


,station_firmware,event_count,percentage
0,v3.2.1,2315839,59.73
1,v3.2.0,1561174,40.27



📅 EVENTS BY MONTH


,month,event_count
0,2024-01,110348
1,2024-02,111531
2,2024-03,123900
3,2024-04,131818
4,2024-05,146278
5,2024-06,148888
6,2024-07,158688
7,2024-08,172295
8,2024-09,195885
9,2024-10,229901


In [ ]:
# STEP 10: Duplicate and near-duplicate audit

print("🔍 DUPLICATE AUDIT")
print("=" * 60)

# 1. Exact duplicate rows
exact_duplicates = swap_events.duplicated().sum()
print("Exact duplicate rows:", exact_duplicates)

# 2. Duplicate event IDs
duplicate_event_ids = swap_events["event_id"].duplicated().sum()
print("Duplicate event IDs:", duplicate_event_ids)

# 3. Offline-batch records
offline = swap_events[
    swap_events["sync_mode"].astype(str).str.lower().eq("offline_batch")
].copy()

print("\nOffline-batch records:", len(offline))

# 4. Check possible near-duplicates
near_dup_cols = [
    "rider_id",
    "station_id",
    "battery_in_id",
    "battery_out_id",
    "event_type"
]

offline = offline.sort_values(
    near_dup_cols + ["event_ts"]
)

offline["time_diff_sec"] = (
    offline.groupby(near_dup_cols)["event_ts"]
    .diff()
    .dt.total_seconds()
)

near_duplicates = offline[
    offline["time_diff_sec"].between(0, 120, inclusive="both")
].copy()

print("Potential near-duplicates within 2 minutes:", len(near_duplicates))

if len(near_duplicates) > 0:
    print("\nSample potential near-duplicates:")
    display(
        near_duplicates[
            near_dup_cols +
            ["event_ts", "time_diff_sec", "sync_mode"]
        ].head(20)
    )


🔍 DUPLICATE AUDIT
Exact duplicate rows: 0
Duplicate event IDs: 0

Offline-batch records: 591114
Potential near-duplicates within 2 minutes: 0


In [ ]:
# STEP 11: Numeric data-quality audit

print("🔍 NUMERIC VALUE AUDIT")
print("=" * 60)

# SOC / SOH values
print("\n1️⃣ SOC / SOH values outside 0–100:")
for col in ["soc_in_pct", "soh_in_pct", "soc_out_pct", "soh_out_pct"]:
    invalid = ((swap_events[col] < 0) | (swap_events[col] > 100)).sum()
    print(f"{col:<15}: {invalid:,}")

# Queue wait
print("\n2️⃣ Negative queue wait:")
print((swap_events["queue_wait_sec"] < 0).sum())

# Distance
print("\n3️⃣ km_since_last_swap:")
print("Negative:", (swap_events["km_since_last_swap"] < 0).sum())
print("Minimum :", swap_events["km_since_last_swap"].min())
print("Maximum :", swap_events["km_since_last_swap"].max())

# Financial values
print("\n4️⃣ Financial values:")
for col in ["list_price_inr", "discount_inr", "amount_charged_inr"]:
    print(
        f"{col:<22} "
        f"Negative: {(swap_events[col] < 0).sum():,} | "
        f"Zero: {(swap_events[col] == 0).sum():,}"
    )

# Energy
print("\n5️⃣ Energy to recharge:")
print("Negative:", (swap_events["energy_to_recharge_kwh"] < 0).sum())
print("Maximum :", swap_events["energy_to_recharge_kwh"].max())

# Show suspicious distance records
print("\n6️⃣ Largest distance values:")
display(
    swap_events[
        ["event_id", "rider_id", "station_id", "km_since_last_swap"]
    ]
    .sort_values("km_since_last_swap", ascending=False)
    .head(10)
)


🔍 NUMERIC VALUE AUDIT

1️⃣ SOC / SOH values outside 0–100:
soc_in_pct     : 0
soh_in_pct     : 3,356
soc_out_pct    : 3,698
soh_out_pct    : 2,799

2️⃣ Negative queue wait:
0

3️⃣ km_since_last_swap:
Negative: 7814
Minimum : -20.0
Maximum : 420.0

4️⃣ Financial values:
list_price_inr         Negative: 0 | Zero: 0
discount_inr           Negative: 0 | Zero: 1,311,535
amount_charged_inr     Negative: 0 | Zero: 231,204

5️⃣ Energy to recharge:
Negative: 0
Maximum : 5.519

6️⃣ Largest distance values:


,event_id,rider_id,station_id,km_since_last_swap
1231109,SW-001231110,RD-1001569,STN-DEL-046,420.00
3075273,SW-003075274,RD-1005225,STN-MUM-113,420.00
2604776,SW-002604777,RD-1005204,STN-MUM-120,420.00
1613197,SW-001613198,RD-1010572,STN-DEL-037,419.90
1730348,SW-001730349,RD-1006567,STN-HYD-076,419.90
1822324,SW-001822325,RD-1008270,STN-DEL-037,419.90
674506,SW-000674507,RD-1002755,STN-HYD-076,419.90
3647775,SW-003647776,RD-1013784,STN-BLR-030,419.90
373422,SW-000373423,RD-1002887,STN-DEL-048,419.90
79383,SW-000079384,RD-1002667,STN-PUN-090,419.90


In [ ]:
# STEP 12: Audit city-name consistency

print("🏙️ CITY NAME AUDIT")
print("=" * 60)

print("\n1️⃣ Rider home_city values:")
rider_cities = (
    riders["home_city"]
    .value_counts(dropna=False)
    .rename_axis("home_city")
    .reset_index(name="rider_count")
)

display(rider_cities)

print("\n2️⃣ Station city values:")
station_cities = (
    stations["city"]
    .value_counts(dropna=False)
    .rename_axis("city")
    .reset_index(name="station_count")
)

display(station_cities)

print("\n3️⃣ City context values:")
context_cities = (
    city_context["city"]
    .value_counts(dropna=False)
    .rename_axis("city")
    .reset_index(name="day_count")
)

display(context_cities)

print("\n4️⃣ Swap-event station IDs vs station master:")
event_station_count = swap_events["station_id"].nunique()
master_station_count = stations["station_id"].nunique()

print("Unique stations in swap events :", event_station_count)
print("Unique stations in station master:", master_station_count)


🏙️ CITY NAME AUDIT

1️⃣ Rider home_city values:


,home_city,rider_count
0,Bengaluru,4175
1,Delhi NCR,3783
2,Hyderabad,3396
3,Pune,2843
4,Mumbai,2772
5,Jaipur,2446
6,Bombay,53
7,BLR,53
8,bengaluru,50
9,MUM,49



2️⃣ Station city values:


,city,station_count
0,Bengaluru,34
1,Delhi NCR,30
2,Hyderabad,26
3,Pune,22
4,Mumbai,22
5,Jaipur,18



3️⃣ City context values:


,city,day_count
0,Bengaluru,547
1,Delhi NCR,547
2,Hyderabad,547
3,Pune,547
4,Mumbai,547
5,Jaipur,547



4️⃣ Swap-event station IDs vs station master:
Unique stations in swap events : 152
Unique stations in station master: 152


In [ ]:
# STEP 13: Standardize city names

city_mapping = {
    # Bengaluru
    "Bangalore": "Bengaluru",
    "BLR": "Bengaluru",
    "bengaluru": "Bengaluru",

    # Mumbai
    "Bombay": "Mumbai",
    "MUM": "Mumbai",
    "mumbai": "Mumbai",

    # Pune
    "pune": "Pune",
    "PUN": "Pune",

    # Delhi
    "delhi": "Delhi",
    "DEL": "Delhi",

    # Hyderabad
    "hyderabad": "Hyderabad",
    "HYD": "Hyderabad",

    # Ahmedabad
    "ahmedabad": "Ahmedabad",
    "AMD": "Ahmedabad"
}

# Keep original columns and create standardized versions
riders["home_city_clean"] = riders["home_city"].replace(city_mapping)
stations["city_clean"] = stations["city"].replace(city_mapping)
city_context["city_clean"] = city_context["city"].replace(city_mapping)

print("🏙️ Standardized city values")
print("=" * 60)

print("\nRider cities:")
display(riders["home_city_clean"].value_counts(dropna=False))

print("\nStation cities:")
display(stations["city_clean"].value_counts(dropna=False))

print("\nCity-context cities:")
display(city_context["city_clean"].value_counts(dropna=False))

print("\n✅ Original city columns have been preserved.")

🏙️ Standardized city values

Rider cities:


,count
home_city_clean,
Bengaluru,4260
Delhi NCR,3783
Hyderabad,3454
Pune,2929
Mumbai,2874
Jaipur,2446
bengaluru,50
New Delhi,42
JAI,41



Station cities:


,count
city_clean,
Bengaluru,34
Delhi NCR,30
Hyderabad,26
Pune,22
Mumbai,22
Jaipur,18



City-context cities:


,count
city_clean,
Bengaluru,547
Delhi NCR,547
Hyderabad,547
Pune,547
Mumbai,547
Jaipur,547



✅ Original city columns have been preserved.


In [ ]:
# STEP 14: Create clean analytical copy

swap_clean = swap_events.copy()

print("🧹 CLEAN DATASET CREATED")
print("=" * 60)
print("Original rows :", len(swap_events))
print("Clean copy rows:", len(swap_clean))

# Create a clean date column
swap_clean["event_date"] = swap_clean["event_ts"].dt.date

# Create month and hour fields for analysis
swap_clean["event_month"] = swap_clean["event_ts"].dt.to_period("M").astype(str)
swap_clean["event_hour"] = swap_clean["event_ts"].dt.hour
swap_clean["day_of_week"] = swap_clean["event_ts"].dt.day_name()

print("\nNew analytical columns:")
print([
    "event_date",
    "event_month",
    "event_hour",
    "day_of_week"
])

print("\n✅ Original swap_events remains unchanged.")

🧹 CLEAN DATASET CREATED
Original rows : 3877013
Clean copy rows: 3877013

New analytical columns:
['event_date', 'event_month', 'event_hour', 'day_of_week']

✅ Original swap_events remains unchanged.


In [ ]:
# STEP 15: Create cleaned numeric variables

print("🧹 CLEANING NUMERIC VARIABLES")
print("=" * 60)

# SOC / SOH: valid range is 0–100
for col in ["soc_in_pct", "soh_in_pct", "soc_out_pct", "soh_out_pct"]:
    clean_col = col + "_clean"
    swap_clean[clean_col] = swap_clean[col].where(
        swap_clean[col].between(0, 100)
    )

# Queue wait: cannot be negative
swap_clean["queue_wait_sec_clean"] = swap_clean["queue_wait_sec"].where(
    swap_clean["queue_wait_sec"] >= 0
)

# Distance: negative distance is invalid
swap_clean["km_since_last_swap_clean"] = swap_clean["km_since_last_swap"].where(
    swap_clean["km_since_last_swap"] >= 0
)

# Financial values: negative values are invalid
for col in [
    "list_price_inr",
    "discount_inr",
    "amount_charged_inr",
    "energy_to_recharge_kwh"
]:
    clean_col = col + "_clean"
    swap_clean[clean_col] = swap_clean[col].where(
        swap_clean[col] >= 0
    )

print("✅ Cleaned columns created.")

print("\nOriginal vs cleaned invalid counts:")
for col in [
    "soc_in_pct",
    "soh_in_pct",
    "soc_out_pct",
    "soh_out_pct",
    "queue_wait_sec",
    "km_since_last_swap",
    "list_price_inr",
    "discount_inr",
    "amount_charged_inr",
    "energy_to_recharge_kwh"
]:
    clean_col = col + "_clean"
    invalid_count = swap_clean[clean_col].isna().sum() - swap_clean[col].isna().sum()
    print(f"{col:<28} {max(0, invalid_count):,}")

🧹 CLEANING NUMERIC VARIABLES
✅ Cleaned columns created.

Original vs cleaned invalid counts:
soc_in_pct                   0
soh_in_pct                   3,356
soc_out_pct                  3,698
soh_out_pct                  2,799
queue_wait_sec               0
km_since_last_swap           7,814
list_price_inr               0
discount_inr                 0
amount_charged_inr           0
energy_to_recharge_kwh       0


In [ ]:
# STEP 16: Create data-quality flags

print("🚦 CREATING DATA-QUALITY FLAGS")
print("=" * 60)

# 1. Flag test station records
swap_clean["is_test_station"] = (
    swap_clean["station_id"].astype(str).str.upper().str.contains("STN-TST", na=False)
)

# 2. Flag duplicate event IDs
swap_clean["is_duplicate_event_id"] = (
    swap_clean["event_id"].duplicated(keep="first")
)

# 3. Flag offline-batch records
swap_clean["is_offline_batch"] = (
    swap_clean["sync_mode"].astype(str).str.lower().eq("offline_batch")
)

# Summary
print("Test-station records       :", swap_clean["is_test_station"].sum())
print("Duplicate event-ID records :", swap_clean["is_duplicate_event_id"].sum())
print("Offline-batch records      :", swap_clean["is_offline_batch"].sum())

print("\nEvent records by test-station flag:")
display(swap_clean["is_test_station"].value_counts())

print("\nEvent records by duplicate flag:")
display(swap_clean["is_duplicate_event_id"].value_counts())

print("\n✅ Quality flags created.")
print("No records have been deleted.")

🚦 CREATING DATA-QUALITY FLAGS
Test-station records       : 62031
Duplicate event-ID records : 0
Offline-batch records      : 591114

Event records by test-station flag:


,count
is_test_station,
False,3814982
True,62031



Event records by duplicate flag:


,count
is_duplicate_event_id,
False,3877013



✅ Quality flags created.
No records have been deleted.


In [ ]:
# STEP 17: Join station information

print("🏪 JOINING STATION INFORMATION")
print("=" * 60)

# Check station master uniqueness
station_duplicates = stations["station_id"].duplicated().sum()
print("Duplicate station IDs in station master:", station_duplicates)

# Select station attributes for the analytical dataset
station_cols = [
    "station_id",
    "city_clean",
    "zone",
    "location_type",
    "host_type",
    "expansion_wave",
    "charger_generation",
    "charger_slots",
    "target_2w_packs",
    "target_3w_packs",
    "monthly_rent_inr",
    "maintenance_monthly_inr",
    "grid_tariff_inr_per_kwh",
    "connectivity_tier",
    "commission_date",
    "decommission_date"
]

station_lookup = stations[station_cols].copy()

# Rename city for clarity after joining
station_lookup = station_lookup.rename(
    columns={"city_clean": "station_city"}
)

# Merge station information
swap_clean = swap_clean.merge(
    station_lookup,
    on="station_id",
    how="left",
    validate="many_to_one"
)

print("\n✅ Station information joined successfully.")

print("Rows after join:", len(swap_clean))

print("\nMissing station information:")
display(
    swap_clean[
        [
            "station_city",
            "zone",
            "location_type",
            "charger_generation",
            "connectivity_tier"
        ]
    ].isna().sum()
)

print("\nStation-level fields added:")
print([
    "station_city",
    "zone",
    "location_type",
    "host_type",
    "expansion_wave",
    "charger_generation",
    "monthly_rent_inr",
    "maintenance_monthly_inr",
    "grid_tariff_inr_per_kwh",
    "connectivity_tier"
])

🏪 JOINING STATION INFORMATION


NameError: name 'stations' is not defined

In [ ]:
# STEP 17A: Check actual station columns

print("🏪 ACTUAL STATION COLUMNS")
print("=" * 60)

for i, col in enumerate(stations.columns, start=1):
    print(f"{i:2}. {col}")

In [ ]:
# STEP 17B: Robust station join using actual columns

print("🏪 JOINING STATION INFORMATION")
print("=" * 60)

# Columns we want if they exist in your dataset
wanted_station_cols = [
    "station_id",
    "city_clean",
    "city",
    "zone",
    "location_type",
    "host_type",
    "expansion_wave",
    "charger_generation",
    "connectivity_tier",
    "monthly_rent_inr",
    "maintenance_monthly_inr",
    "grid_tariff_inr_per_kwh"
]

# Keep only columns that actually exist
available_station_cols = [
    col for col in wanted_station_cols
    if col in stations.columns
]

missing_station_cols = [
    col for col in wanted_station_cols
    if col not in stations.columns
]

print("Columns available for join:")
print(available_station_cols)

print("\nColumns not found:")
print(missing_station_cols)

# Make lookup table
station_lookup = stations[available_station_cols].copy()

# Make sure station_id is unique
print("\nDuplicate station IDs:",
      station_lookup["station_id"].duplicated().sum())

# Avoid duplicate city column if both exist
if "city_clean" in station_lookup.columns:
    station_lookup = station_lookup.rename(
        columns={"city_clean": "station_city"}
    )
elif "city" in station_lookup.columns:
    station_lookup = station_lookup.rename(
        columns={"city": "station_city"}
    )

# Merge
swap_clean = swap_clean.merge(
    station_lookup,
    on="station_id",
    how="left",
    validate="many_to_one"
)

print("\n✅ Station information joined successfully.")
print("Rows after join:", len(swap_clean))

print("\nSample joined data:")
display(
    swap_clean[
        [col for col in [
            "station_id",
            "station_city",
            "zone",
            "location_type",
            "charger_generation",
            "connectivity_tier"
        ] if col in swap_clean.columns]
    ].head()
)

In [ ]:
# STEP 18: Join rider information

print("👤 JOINING RIDER INFORMATION")
print("=" * 60)

# Select only columns that actually exist
wanted_rider_cols = [
    "rider_id",
    "partner_id",
    "vehicle_class",
    "vehicle_model",
    "home_city_clean",
    "home_city",
    "home_zone",
    "signup_date",
    "signup_channel",
    "plan_type",
    "declared_shift",
    "age_band",
    "kyc_verified"
]

available_rider_cols = [
    col for col in wanted_rider_cols
    if col in riders.columns
]

print("Columns being joined:")
print(available_rider_cols)

# Create lookup table
rider_lookup = riders[available_rider_cols].copy()

# Check rider ID uniqueness
print("\nDuplicate rider IDs:",
      rider_lookup["rider_id"].duplicated().sum())

# Prefer standardized city if available
if "home_city_clean" in rider_lookup.columns:
    rider_lookup = rider_lookup.rename(
        columns={"home_city_clean": "rider_city"}
    )
elif "home_city" in rider_lookup.columns:
    rider_lookup = rider_lookup.rename(
        columns={"home_city": "rider_city"}
    )

# Merge rider information
swap_clean = swap_clean.merge(
    rider_lookup,
    on="rider_id",
    how="left",
    validate="many_to_one"
)

print("\n✅ Rider information joined successfully.")
print("Rows after join:", len(swap_clean))

print("\nSample joined data:")
display(
    swap_clean[
        [col for col in [
            "rider_id",
            "partner_id",
            "vehicle_class",
            "vehicle_model",
            "rider_city",
            "home_zone",
            "plan_type",
            "signup_channel"
        ] if col in swap_clean.columns]
    ].head()
)

print("\nMissing rider IDs after join:",
      swap_clean["partner_id"].isna().sum())

In [ ]:
# STEP 19: Join battery information

print("🔋 JOINING BATTERY INFORMATION")
print("=" * 60)

# Keep useful battery columns that actually exist
wanted_battery_cols = [
    "battery_id",
    "pack_type",
    "supplier",
    "manufacturing_lot",
    "manufacture_date",
    "commission_date",
    "rated_capacity_kwh",
    "purchase_cost_inr",
    "initial_soh_pct",
    "bms_firmware",
    "current_soh_pct",
    "retired_date",
    "retired_reason"
]

available_battery_cols = [
    col for col in wanted_battery_cols
    if col in batteries.columns
]

print("Battery columns available:")
print(available_battery_cols)

# Check battery master uniqueness
print("\nDuplicate battery IDs:",
      batteries["battery_id"].duplicated().sum())

# Create battery lookup
battery_lookup = batteries[available_battery_cols].copy()

# ---------------------------------------------------------
# Battery IN information
# ---------------------------------------------------------

battery_in_lookup = battery_lookup.copy()

battery_in_lookup = battery_in_lookup.rename(
    columns={
        col: f"battery_in_{col}"
        for col in available_battery_cols
        if col != "battery_id"
    }
)

swap_clean = swap_clean.merge(
    battery_in_lookup,
    left_on="battery_in_id",
    right_on="battery_id",
    how="left",
    validate="many_to_one"
)

# Remove duplicate lookup key
if "battery_id" in swap_clean.columns:
    swap_clean = swap_clean.drop(columns=["battery_id"])

# ---------------------------------------------------------
# Battery OUT information
# ---------------------------------------------------------

battery_out_lookup = battery_lookup.copy()

battery_out_lookup = battery_out_lookup.rename(
    columns={
        col: f"battery_out_{col}"
        for col in available_battery_cols
        if col != "battery_id"
    }
)

swap_clean = swap_clean.merge(
    battery_out_lookup,
    left_on="battery_out_id",
    right_on="battery_id",
    how="left",
    validate="many_to_one"
)

if "battery_id" in swap_clean.columns:
    swap_clean = swap_clean.drop(columns=["battery_id"])

print("\n✅ Battery information joined successfully.")
print("Rows after joins:", len(swap_clean))

print("\nSample battery information:")
display(
    swap_clean[
        [col for col in [
            "battery_in_id",
            "battery_in_supplier",
            "battery_in_current_soh_pct",
            "battery_out_id",
            "battery_out_supplier",
            "battery_out_current_soh_pct"
        ] if col in swap_clean.columns]
    ].head()
)

In [ ]:
# STEP 20: Join daily city context

print("🌦️ JOINING CITY DAILY CONTEXT")
print("=" * 60)

# Check the actual city-context columns
print("City context columns:")
print(city_context.columns.tolist())

# Make a working copy
context_lookup = city_context.copy()

# Identify the date column
if "date" in context_lookup.columns:
    context_date_col = "date"
elif "context_date" in context_lookup.columns:
    context_date_col = "context_date"
else:
    raise ValueError("Could not find the date column in city_daily_context.csv")

# Convert date to datetime
context_lookup[context_date_col] = pd.to_datetime(
    context_lookup[context_date_col],
    errors="coerce"
).dt.date

# Create event date in the same format
swap_clean["event_date_join"] = pd.to_datetime(
    swap_clean["event_ts"]
).dt.date

# Make sure city context has standardized city
if "city_clean" in context_lookup.columns:
    context_lookup = context_lookup.rename(
        columns={"city_clean": "context_city"}
    )
elif "city" in context_lookup.columns:
    context_lookup = context_lookup.rename(
        columns={"city": "context_city"}
    )
else:
    raise ValueError("Could not find the city column in city_daily_context.csv")

# Check uniqueness of city + date
duplicate_context = context_lookup.duplicated(
    subset=["context_city", context_date_col]
).sum()

print("\nDuplicate city-date records:", duplicate_context)

# Rename date for joining
context_lookup = context_lookup.rename(
    columns={context_date_col: "event_date_join"}
)

# Avoid duplicate columns
context_cols = [
    col for col in context_lookup.columns
    if col not in ["event_date_join", "context_city"]
]

# Join city + date
swap_clean = swap_clean.merge(
    context_lookup[
        ["context_city", "event_date_join"] + context_cols
    ],
    left_on=["station_city", "event_date_join"],
    right_on=["context_city", "event_date_join"],
    how="left",
    validate="many_to_one"
)

print("\n✅ City context joined successfully.")
print("Rows after join:", len(swap_clean))

print("\nMissing city-context matches:")
print(
    swap_clean[context_cols]
    .isna()
    .all(axis=1)
    .sum()
)

print("\nSample joined data:")
display(
    swap_clean[
        ["event_ts", "station_city", "event_date_join"] +
        context_cols[:6]
    ].head()
)

In [ ]:
# STEP 20A: Verify city-context join

print("✅ VERIFYING CITY CONTEXT JOIN")
print("=" * 60)

# Show columns added from city context
context_keywords = [
    "temp",
    "rain",
    "heat",
    "flood",
    "festival",
    "holiday",
    "grid",
    "competitor",
    "sale"
]

context_added = [
    col for col in swap_clean.columns
    if any(keyword in col.lower() for keyword in context_keywords)
]

print("City-context columns found:")
print(context_added)

print("\nNumber of rows:", len(swap_clean))

print("\nSample:")
display(
    swap_clean[
        ["event_ts", "station_city", "event_date_join"] +
        context_added[:6]
    ].head()
)


In [ ]:
# STEP 21: Join station-hourly operational data

print("⚡ JOINING STATION-HOURLY DATA")
print("=" * 60)

# Check actual columns
print("Station-hourly columns:")
print(station_hourly.columns.tolist())

# Create a working copy
hourly_lookup = station_hourly.copy()

# Find the timestamp column automatically
possible_time_cols = ["hour_ts", "timestamp", "event_ts", "hour"]

time_col = next(
    (col for col in possible_time_cols if col in hourly_lookup.columns),
    None
)

if time_col is None:
    raise ValueError(
        "Could not find the hourly timestamp column. "
        "Please check the columns printed above."
    )

print("\nUsing timestamp column:", time_col)

# Convert to datetime
hourly_lookup[time_col] = pd.to_datetime(
    hourly_lookup[time_col],
    errors="coerce"
)

# Create matching hourly timestamp
swap_clean["event_hour_ts"] = swap_clean["event_ts"].dt.floor("h")

hourly_lookup["hour_key"] = hourly_lookup[time_col].dt.floor("h")

# Check uniqueness
duplicate_hourly = hourly_lookup.duplicated(
    subset=["station_id", "hour_key"]
).sum()

print("Duplicate station-hour records:", duplicate_hourly)

# Operational columns
hourly_cols = [
    col for col in hourly_lookup.columns
    if col not in ["station_id", time_col, "hour_key"]
]

# Avoid columns that already exist in swap_clean
hourly_cols = [
    col for col in hourly_cols
    if col not in swap_clean.columns
]

# Join
swap_clean = swap_clean.merge(
    hourly_lookup[
        ["station_id", "hour_key"] + hourly_cols
    ],
    left_on=["station_id", "event_hour_ts"],
    right_on=["station_id", "hour_key"],
    how="left",
    validate="many_to_one"
)

# Remove temporary key
swap_clean = swap_clean.drop(columns=["hour_key"])

print("\n✅ Station-hourly data joined successfully.")
print("Rows after join:", len(swap_clean))

# Match rate
if hourly_cols:
    matched = swap_clean[hourly_cols].notna().any(axis=1).sum()
    match_rate = matched / len(swap_clean) * 100

    print(
        "Events matched with station-hour data:",
        f"{matched:,}"
    )
    print(
        "Match rate:",
        f"{match_rate:.2f}%"
    )

print("\nOperational columns added:")
print(hourly_cols)

In [ ]:
# STEP 21A: Inspect station-hourly dataset

print("⚡ STATION-HOURLY DATA CHECK")
print("=" * 60)

print("Shape:", station_hourly.shape)

print("\nColumns:")
for i, col in enumerate(station_hourly.columns, 1):
    print(f"{i:2}. {col}")

print("\nFirst 3 rows:")
display(station_hourly.head(3))

print("\nData types:")
display(station_hourly.dtypes)

In [ ]:
# STEP 21A: Safe station-hourly structure check

print("⚡ STATION-HOURLY DATA CHECK")
print("=" * 60)

print("Dataset shape:", station_hourly.shape)

print("\nColumns:")
print(station_hourly.columns.tolist())

print("\nPossible time/date columns:")

possible_time_cols = [
    col for col in station_hourly.columns
    if any(word in col.lower() for word in [
        "time", "date", "hour", "timestamp", "ts"
    ])
]

for col in possible_time_cols:
    print(
        f"- {col} | dtype: {station_hourly[col].dtype} | "
        f"sample: {station_hourly[col].iloc[0]}"
    )

print("\nFirst 3 rows:")
display(station_hourly.head(3))

In [ ]:
# STEP 21: Robust station-hourly join

print("⚡ STATION-HOURLY JOIN")
print("=" * 60)

# Start fresh
hourly_lookup = station_hourly.copy()

# ---------------------------------------------------------
# 1. Find the station ID column
# ---------------------------------------------------------
if "station_id" not in hourly_lookup.columns:
    raise ValueError(
        "station_id was not found in station_hourly. "
        f"Available columns: {hourly_lookup.columns.tolist()}"
    )

# ---------------------------------------------------------
# 2. Automatically find the best timestamp column
# ---------------------------------------------------------
time_candidates = [
    col for col in hourly_lookup.columns
    if any(word in col.lower() for word in
           ["timestamp", "time", "date", "hour", "ts"])
]

print("Possible time columns:", time_candidates)

best_time_col = None
best_valid_count = -1

for col in time_candidates:
    converted = pd.to_datetime(
        hourly_lookup[col],
        errors="coerce"
    )
    valid_count = converted.notna().sum()

    if valid_count > best_valid_count:
        best_valid_count = valid_count
        best_time_col = col

if best_time_col is None:
    raise ValueError(
        "No usable timestamp column was found."
    )

print("Selected timestamp column:", best_time_col)
print("Valid timestamps:", f"{best_valid_count:,}")

# ---------------------------------------------------------
# 3. Prepare timestamps
# ---------------------------------------------------------
hourly_lookup["__hour_key"] = (
    pd.to_datetime(
        hourly_lookup[best_time_col],
        errors="coerce"
    ).dt.floor("h")
)

swap_clean["event_hour_ts"] = (
    pd.to_datetime(
        swap_clean["event_ts"],
        errors="coerce"
    ).dt.floor("h")
)

# ---------------------------------------------------------
# 4. Check station-hour uniqueness
# ---------------------------------------------------------
duplicates = hourly_lookup.duplicated(
    subset=["station_id", "__hour_key"]
).sum()

print("Duplicate station-hour records:", duplicates)

# If duplicates exist, keep the first record for each station-hour
if duplicates > 0:
    hourly_lookup = hourly_lookup.drop_duplicates(
        subset=["station_id", "__hour_key"],
        keep="first"
    )

# ---------------------------------------------------------
# 5. Identify operational columns
# ---------------------------------------------------------
exclude_cols = {
    "station_id",
    best_time_col,
    "__hour_key"
}

hourly_cols = [
    col for col in hourly_lookup.columns
    if col not in exclude_cols
]

# Don't overwrite columns already in swap_clean
hourly_cols = [
    col for col in hourly_cols
    if col not in swap_clean.columns
]

print("\nOperational columns to add:")
print(hourly_cols)

# ---------------------------------------------------------
# 6. Perform the join
# ---------------------------------------------------------
swap_clean = swap_clean.merge(
    hourly_lookup[
        ["station_id", "__hour_key"] + hourly_cols
    ],
    left_on=["station_id", "event_hour_ts"],
    right_on=["station_id", "__hour_key"],
    how="left",
    validate="many_to_one"
)

# Remove temporary join key
swap_clean.drop(
    columns=["__hour_key"],
    inplace=True,
    errors="ignore"
)

# ---------------------------------------------------------
# 7. Check result
# ---------------------------------------------------------
print("\n✅ Station-hourly join completed!")
print("Rows in analytical dataset:", f"{len(swap_clean):,}")

if hourly_cols:
    matched = swap_clean[hourly_cols].notna().any(axis=1).sum()
    match_rate = matched / len(swap_clean) * 100

    print("Matched events:", f"{matched:,}")
    print("Match rate:", f"{match_rate:.2f}%")

print("\nFinal dataset columns:", len(swap_clean.columns))

In [ ]:
# STEP 22: Final analytical dataset check

print("📊 FINAL ANALYTICAL DATASET CHECK")
print("=" * 65)

print("Rows:", f"{len(swap_clean):,}")
print("Columns:", len(swap_clean.columns))

print("\nExpected original swap rows: 1,012,450")

if len(swap_clean) == len(swap_events):
    print("✅ Row count preserved")
else:
    print("⚠️ Row count changed — investigate before analysis")

print("\nMissing values in key analysis fields:")
key_columns = [
    "event_ts",
    "event_type",
    "station_id",
    "rider_id",
    "battery_in_id",
    "battery_out_id",
    "amount_charged_inr",
    "queue_wait_sec",
    "station_city",
    "vehicle_class"
]

available_key_columns = [
    col for col in key_columns
    if col in swap_clean.columns
]

display(
    swap_clean[available_key_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nEvent types:")
display(swap_clean["event_type"].value_counts())

print("\nDate range:")
print(
    swap_clean["event_ts"].min(),
    "→",
    swap_clean["event_ts"].max()
)

print("\n✅ DATA PREPARATION COMPLETE")

In [ ]:
# STEP 23: Monthly Network Performance KPIs

print("📈 NETWORK PERFORMANCE — MONTHLY KPIs")
print("=" * 70)

# Create success/failure flags
swap_clean["is_completed"] = (
    swap_clean["event_type"] == "swap_completed"
)

swap_clean["is_failed"] = (
    swap_clean["event_type"].isin([
        "failed_no_charged_battery",
        "failed_system_error"
    ])
)

swap_clean["is_abandoned"] = (
    swap_clean["event_type"] == "abandoned_queue"
)

# Monthly aggregation
monthly_kpi = (
    swap_clean
    .groupby("event_month")
    .agg(
        total_attempts=("event_id", "count"),
        completed_swaps=("is_completed", "sum"),
        failed_attempts=("is_failed", "sum"),
        abandoned_swaps=("is_abandoned", "sum"),
        revenue_inr=("amount_charged_inr", "sum")
    )
    .reset_index()
)

# Calculate rates
monthly_kpi["completion_rate_pct"] = (
    monthly_kpi["completed_swaps"]
    / monthly_kpi["total_attempts"] * 100
)

monthly_kpi["failure_rate_pct"] = (
    monthly_kpi["failed_attempts"]
    / monthly_kpi["total_attempts"] * 100
)

monthly_kpi["abandonment_rate_pct"] = (
    monthly_kpi["abandoned_swaps"]
    / monthly_kpi["total_attempts"] * 100
)

monthly_kpi["revenue_per_completed_swap_inr"] = (
    monthly_kpi["revenue_inr"]
    / monthly_kpi["completed_swaps"]
)

# Display
display(
    monthly_kpi.style.format({
        "total_attempts": "{:,.0f}",
        "completed_swaps": "{:,.0f}",
        "failed_attempts": "{:,.0f}",
        "abandoned_swaps": "{:,.0f}",
        "revenue_inr": "₹{:,.0f}",
        "completion_rate_pct": "{:.2f}%",
        "failure_rate_pct": "{:.2f}%",
        "abandonment_rate_pct": "{:.2f}%",
        "revenue_per_completed_swap_inr": "₹{:,.2f}"
    })
)

print("\n✅ Monthly KPI table created.")

In [ ]:
# STEP 24: Network Performance Trend Charts

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Completed swaps
axes[0, 0].plot(
    monthly_kpi["event_month"],
    monthly_kpi["completed_swaps"],
    marker="o"
)
axes[0, 0].set_title("Completed Swaps by Month")
axes[0, 0].set_xlabel("Month")
axes[0, 0].set_ylabel("Completed Swaps")
axes[0, 0].tick_params(axis="x", rotation=45)

# 2. Failure rate
axes[0, 1].plot(
    monthly_kpi["event_month"],
    monthly_kpi["failure_rate_pct"],
    marker="o"
)
axes[0, 1].set_title("Failure Rate by Month")
axes[0, 1].set_xlabel("Month")
axes[0, 1].set_ylabel("Failure Rate (%)")
axes[0, 1].tick_params(axis="x", rotation=45)

# 3. Revenue
axes[1, 0].plot(
    monthly_kpi["event_month"],
    monthly_kpi["revenue_inr"],
    marker="o"
)
axes[1, 0].set_title("Revenue by Month")
axes[1, 0].set_xlabel("Month")
axes[1, 0].set_ylabel("Revenue (₹)")
axes[1, 0].tick_params(axis="x", rotation=45)

# 4. Revenue per completed swap
axes[1, 1].plot(
    monthly_kpi["event_month"],
    monthly_kpi["revenue_per_completed_swap_inr"],
    marker="o"
)
axes[1, 1].set_title("Revenue per Completed Swap")
axes[1, 1].set_xlabel("Month")
axes[1, 1].set_ylabel("₹ per completed swap")
axes[1, 1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# STEP 25: Month-over-month changes in key network KPIs

monthly_kpi = monthly_kpi.sort_values("event_month").copy()

kpi_cols = [
    "completed_swaps",
    "failure_rate_pct",
    "revenue_inr",
    "revenue_per_completed_swap_inr"
]

for col in kpi_cols:
    monthly_kpi[f"{col}_mom_pct"] = monthly_kpi[col].pct_change() * 100

display(monthly_kpi)

In [ ]:
# STEP 26: Failure and abandonment patterns by city

city_service = (
    swap_clean
    .groupby("station_city")
    .agg(
        total_attempts=("event_id", "count"),
        completed_swaps=("is_completed", "sum"),
        failed_attempts=("is_failed", "sum"),
        abandoned_swaps=("is_abandoned", "sum"),
        avg_queue_wait_min=("queue_wait_min_clean", "mean")
    )
    .reset_index()
)

city_service["completion_rate_pct"] = (
    city_service["completed_swaps"] /
    city_service["total_attempts"] * 100
)

city_service["failure_rate_pct"] = (
    city_service["failed_attempts"] /
    city_service["total_attempts"] * 100
)

city_service["abandonment_rate_pct"] = (
    city_service["abandoned_swaps"] /
    city_service["total_attempts"] * 100
)

city_service = city_service.sort_values(
    "failure_rate_pct", ascending=False
)

display(city_service.round(2))

In [ ]:
# STEP 26: Service failure patterns by city — robust version

# Check which city column is available
if "station_city" in swap_clean.columns:
    city_col = "station_city"
elif "city_clean" in swap_clean.columns:
    city_col = "city_clean"
elif "city" in swap_clean.columns:
    city_col = "city"
else:
    raise KeyError("No city column found in swap_clean.")

# Create service flags if they are not already present
swap_clean["is_completed"] = (
    swap_clean["event_type"] == "swap_completed"
)

swap_clean["is_failed"] = swap_clean["event_type"].isin([
    "failed_no_charged_battery",
    "failed_system_error"
])

swap_clean["is_abandoned"] = (
    swap_clean["event_type"] == "abandoned_queue"
)

# Aggregate by city
city_service = (
    swap_clean
    .groupby(city_col)
    .agg(
        total_attempts=("event_type", "size"),
        completed_swaps=("is_completed", "sum"),
        failed_attempts=("is_failed", "sum"),
        abandoned_swaps=("is_abandoned", "sum")
    )
    .reset_index()
)

# Average queue wait, if available
if "queue_wait_min_clean" in swap_clean.columns:
    queue_avg = (
        swap_clean.groupby(city_col)["queue_wait_min_clean"]
        .mean()
        .reset_index(name="avg_queue_wait_min")
    )
    city_service = city_service.merge(
        queue_avg, on=city_col, how="left"
    )

# Calculate rates
city_service["completion_rate_pct"] = (
    city_service["completed_swaps"] /
    city_service["total_attempts"] * 100
)

city_service["failure_rate_pct"] = (
    city_service["failed_attempts"] /
    city_service["total_attempts"] * 100
)

city_service["abandonment_rate_pct"] = (
    city_service["abandoned_swaps"] /
    city_service["total_attempts"] * 100
)

# Sort by failure rate
city_service = city_service.sort_values(
    "failure_rate_pct",
    ascending=False
)

display(city_service.round(2))

In [ ]:
# STEP 27: Failure and abandonment patterns by hour

hourly_service = (
    swap_clean
    .groupby("event_hour")
    .agg(
        total_attempts=("event_type", "size"),
        completed_swaps=("is_completed", "sum"),
        failed_attempts=("is_failed", "sum"),
        abandoned_swaps=("is_abandoned", "sum"),
        avg_queue_wait_min=("queue_wait_min_clean", "mean")
    )
    .reset_index()
)

# Calculate service rates
hourly_service["completion_rate_pct"] = (
    hourly_service["completed_swaps"] /
    hourly_service["total_attempts"] * 100
)

hourly_service["failure_rate_pct"] = (
    hourly_service["failed_attempts"] /
    hourly_service["total_attempts"] * 100
)

hourly_service["abandonment_rate_pct"] = (
    hourly_service["abandoned_swaps"] /
    hourly_service["total_attempts"] * 100
)

# Display in chronological order
hourly_service = hourly_service.sort_values("event_hour")

display(hourly_service.round(2))

In [ ]:
# STEP 27: Failure and abandonment patterns by hour — fixed

# Make sure event_ts is datetime
swap_clean["event_ts"] = pd.to_datetime(
    swap_clean["event_ts"],
    errors="coerce"
)

# Create hour directly from timestamp
swap_clean["analysis_hour"] = swap_clean["event_ts"].dt.hour

# Recreate service flags
swap_clean["is_completed"] = (
    swap_clean["event_type"] == "swap_completed"
)

swap_clean["is_failed"] = swap_clean["event_type"].isin([
    "failed_no_charged_battery",
    "failed_system_error"
])

swap_clean["is_abandoned"] = (
    swap_clean["event_type"] == "abandoned_queue"
)

# Aggregate by hour
hourly_service = (
    swap_clean
    .dropna(subset=["analysis_hour"])
    .groupby("analysis_hour")
    .agg(
        total_attempts=("event_type", "size"),
        completed_swaps=("is_completed", "sum"),
        failed_attempts=("is_failed", "sum"),
        abandoned_swaps=("is_abandoned", "sum")
    )
    .reset_index()
)

# Add average queue wait if the cleaned column exists
if "queue_wait_min_clean" in swap_clean.columns:
    queue_data = (
        swap_clean
        .dropna(subset=["analysis_hour"])
        .groupby("analysis_hour")["queue_wait_min_clean"]
        .mean()
        .reset_index(name="avg_queue_wait_min")
    )

    hourly_service = hourly_service.merge(
        queue_data,
        on="analysis_hour",
        how="left"
    )

# Calculate percentages
hourly_service["completion_rate_pct"] = (
    hourly_service["completed_swaps"] /
    hourly_service["total_attempts"] * 100
)

hourly_service["failure_rate_pct"] = (
    hourly_service["failed_attempts"] /
    hourly_service["total_attempts"] * 100
)

hourly_service["abandonment_rate_pct"] = (
    hourly_service["abandoned_swaps"] /
    hourly_service["total_attempts"] * 100
)

# Sort chronologically
hourly_service = hourly_service.sort_values("analysis_hour")

display(hourly_service.round(2))

In [ ]:
# STEP 28: Identify high-risk service hours

# Top hours by failure rate
top_failure_hours = (
    hourly_service
    .sort_values("failure_rate_pct", ascending=False)
    .head(5)
)

# Top hours by abandonment rate
top_abandonment_hours = (
    hourly_service
    .sort_values("abandonment_rate_pct", ascending=False)
    .head(5)
)

print("TOP 5 HOURS BY FAILURE RATE")
display(
    top_failure_hours[
        [
            "analysis_hour",
            "total_attempts",
            "failed_attempts",
            "failure_rate_pct",
            "avg_queue_wait_min"
        ]
    ].round(2)
)

print("\nTOP 5 HOURS BY ABANDONMENT RATE")
display(
    top_abandonment_hours[
        [
            "analysis_hour",
            "total_attempts",
            "abandoned_swaps",
            "abandonment_rate_pct",
            "avg_queue_wait_min"
        ]
    ].round(2)
)

In [ ]:
# STEP 28: Identify high-risk service hours — fixed

# Make sure the required columns exist
required_cols = [
    "analysis_hour",
    "total_attempts",
    "failed_attempts",
    "failure_rate_pct",
    "abandoned_swaps",
    "abandonment_rate_pct"
]

missing_cols = [
    col for col in required_cols
    if col not in hourly_service.columns
]

if missing_cols:
    print("Missing columns:", missing_cols)
    print("\nAvailable columns:")
    print(hourly_service.columns.tolist())
else:
    # Top 5 hours by failure rate
    top_failure_hours = (
        hourly_service
        .sort_values("failure_rate_pct", ascending=False)
        .head(5)
    )

    # Top 5 hours by abandonment rate
    top_abandonment_hours = (
        hourly_service
        .sort_values("abandonment_rate_pct", ascending=False)
        .head(5)
    )

    print("TOP 5 HOURS BY FAILURE RATE")
    display(
        top_failure_hours[
            [
                "analysis_hour",
                "total_attempts",
                "failed_attempts",
                "failure_rate_pct"
            ]
        ].round(2)
    )

    print("\nTOP 5 HOURS BY ABANDONMENT RATE")
    display(
        top_abandonment_hours[
            [
                "analysis_hour",
                "total_attempts",
                "abandoned_swaps",
                "abandonment_rate_pct"
            ]
        ].round(2)
    )

In [ ]:
# STEP 29: Station-level service performance

# Find the station column
if "station_id" in swap_clean.columns:
    station_col = "station_id"
else:
    raise KeyError("station_id column not found in swap_clean.")

# Calculate station-level KPIs
station_service = (
    swap_clean
    .groupby(station_col)
    .agg(
        total_attempts=("event_type", "size"),
        completed_swaps=("is_completed", "sum"),
        failed_attempts=("is_failed", "sum"),
        abandoned_swaps=("is_abandoned", "sum")
    )
    .reset_index()
)

# Calculate rates
station_service["completion_rate_pct"] = (
    station_service["completed_swaps"] /
    station_service["total_attempts"] * 100
)

station_service["failure_rate_pct"] = (
    station_service["failed_attempts"] /
    station_service["total_attempts"] * 100
)

station_service["abandonment_rate_pct"] = (
    station_service["abandoned_swaps"] /
    station_service["total_attempts"] * 100
)

# Remove very low-volume stations from the ranking
# so that a station with only a few attempts doesn't dominate
station_service_filtered = station_service[
    station_service["total_attempts"] >= 100
].copy()

# Highest failure-rate stations
top_failure_stations = (
    station_service_filtered
    .sort_values("failure_rate_pct", ascending=False)
    .head(10)
)

print("TOP 10 STATIONS BY FAILURE RATE")
display(
    top_failure_stations.round(2)
)

In [ ]:
# STEP 30: Service performance by charger generation

# Check that charger generation exists
if "charger_generation" not in swap_clean.columns:
    print("charger_generation column not found.")
    print("Available columns:")
    print(swap_clean.columns.tolist())
else:
    charger_service = (
        swap_clean
        .groupby("charger_generation")
        .agg(
            total_attempts=("event_type", "size"),
            completed_swaps=("is_completed", "sum"),
            failed_attempts=("is_failed", "sum"),
            abandoned_swaps=("is_abandoned", "sum")
        )
        .reset_index()
    )

    # Calculate service rates
    charger_service["completion_rate_pct"] = (
        charger_service["completed_swaps"] /
        charger_service["total_attempts"] * 100
    )

    charger_service["failure_rate_pct"] = (
        charger_service["failed_attempts"] /
        charger_service["total_attempts"] * 100
    )

    charger_service["abandonment_rate_pct"] = (
        charger_service["abandoned_swaps"] /
        charger_service["total_attempts"] * 100
    )

    charger_service = charger_service.sort_values(
        "failure_rate_pct",
        ascending=False
    )

    display(charger_service.round(2))

In [ ]:
# STEP 31: Service performance by battery supplier

if "battery_in_supplier" not in swap_clean.columns:
    print("battery_in_supplier column not found.")
    print("Available battery columns:")
    print([
        c for c in swap_clean.columns
        if "supplier" in c.lower()
    ])
else:
    supplier_service = (
        swap_clean
        .groupby("battery_in_supplier")
        .agg(
            total_attempts=("event_type", "size"),
            completed_swaps=("is_completed", "sum"),
            failed_attempts=("is_failed", "sum"),
            abandoned_swaps=("is_abandoned", "sum")
        )
        .reset_index()
    )

    supplier_service["completion_rate_pct"] = (
        supplier_service["completed_swaps"] /
        supplier_service["total_attempts"] * 100
    )

    supplier_service["failure_rate_pct"] = (
        supplier_service["failed_attempts"] /
        supplier_service["total_attempts"] * 100
    )

    supplier_service["abandonment_rate_pct"] = (
        supplier_service["abandoned_swaps"] /
        supplier_service["total_attempts"] * 100
    )

    supplier_service = supplier_service.sort_values(
        "failure_rate_pct",
        ascending=False
    )

    display(supplier_service.round(2))

In [ ]:
# STEP 32: Battery SOH bands vs service outcomes

soh_col = "battery_in_current_soh_pct"

if soh_col not in swap_clean.columns:
    print("SOH column not found.")
    print("Available SOH columns:")
    print([
        c for c in swap_clean.columns
        if "soh" in c.lower()
    ])
else:
    # Create SOH bands
    swap_clean["soh_band"] = pd.cut(
        swap_clean[soh_col],
        bins=[0, 60, 70, 80, 90, 100],
        labels=[
            "<60%",
            "60–70%",
            "70–80%",
            "80–90%",
            "90–100%"
        ],
        include_lowest=True
    )

    soh_service = (
        swap_clean
        .dropna(subset=["soh_band"])
        .groupby("soh_band", observed=False)
        .agg(
            total_attempts=("event_type", "size"),
            completed_swaps=("is_completed", "sum"),
            failed_attempts=("is_failed", "sum"),
            abandoned_swaps=("is_abandoned", "sum"),
            avg_soh_pct=(soh_col, "mean")
        )
        .reset_index()
    )

    soh_service["completion_rate_pct"] = (
        soh_service["completed_swaps"] /
        soh_service["total_attempts"] * 100
    )

    soh_service["failure_rate_pct"] = (
        soh_service["failed_attempts"] /
        soh_service["total_attempts"] * 100
    )

    soh_service["abandonment_rate_pct"] = (
        soh_service["abandoned_swaps"] /
        soh_service["total_attempts"] * 100
    )

    display(soh_service.round(2))

In [ ]:
# STEP 33: Battery age vs service outcomes

date_col = "battery_in_commission_date"

if date_col not in swap_clean.columns:
    print("Battery commission date column not found.")
    print("Available date columns:")
    print([
        c for c in swap_clean.columns
        if "date" in c.lower() and "battery_in" in c.lower()
    ])
else:
    # Convert dates
    swap_clean["battery_commission_date"] = pd.to_datetime(
        swap_clean[date_col],
        errors="coerce"
    )

    # Calculate battery age in days at time of swap
    swap_clean["battery_age_days"] = (
        swap_clean["event_ts"] -
        swap_clean["battery_commission_date"]
    ).dt.total_seconds() / (60 * 60 * 24)

    # Remove impossible negative ages
    swap_clean.loc[
        swap_clean["battery_age_days"] < 0,
        "battery_age_days"
    ] = np.nan

    # Create practical age bands
    swap_clean["battery_age_band"] = pd.cut(
        swap_clean["battery_age_days"],
        bins=[-1, 90, 180, 365, 730, np.inf],
        labels=[
            "<3 months",
            "3–6 months",
            "6–12 months",
            "1–2 years",
            "2+ years"
        ]
    )

    battery_age_service = (
        swap_clean
        .dropna(subset=["battery_age_band"])
        .groupby("battery_age_band", observed=False)
        .agg(
            total_attempts=("event_type", "size"),
            completed_swaps=("is_completed", "sum"),
            failed_attempts=("is_failed", "sum"),
            abandoned_swaps=("is_abandoned", "sum"),
            avg_battery_age_days=("battery_age_days", "mean")
        )
        .reset_index()
    )

    battery_age_service["completion_rate_pct"] = (
        battery_age_service["completed_swaps"] /
        battery_age_service["total_attempts"] * 100
    )

    battery_age_service["failure_rate_pct"] = (
        battery_age_service["failed_attempts"] /
        battery_age_service["total_attempts"] * 100
    )

    battery_age_service["abandonment_rate_pct"] = (
        battery_age_service["abandoned_swaps"] /
        battery_age_service["total_attempts"] * 100
    )

    display(battery_age_service.round(2))

In [ ]:
# STEP 34: Pricing and revenue performance by vehicle class

if "vehicle_class" not in swap_clean.columns:
    print("vehicle_class column not found.")
    print("Available vehicle-related columns:")
    print([
        c for c in swap_clean.columns
        if "vehicle" in c.lower()
    ])
else:
    vehicle_economics = (
        swap_clean
        .groupby("vehicle_class")
        .agg(
            total_attempts=("event_type", "size"),
            completed_swaps=("is_completed", "sum"),
            failed_attempts=("is_failed", "sum"),
            revenue_inr=("amount_charged_clean", "sum"),
            avg_amount_charged_inr=("amount_charged_clean", "mean")
        )
        .reset_index()
    )

    vehicle_economics["completion_rate_pct"] = (
        vehicle_economics["completed_swaps"] /
        vehicle_economics["total_attempts"] * 100
    )

    vehicle_economics["failure_rate_pct"] = (
        vehicle_economics["failed_attempts"] /
        vehicle_economics["total_attempts"] * 100
    )

    vehicle_economics["revenue_per_attempt_inr"] = (
        vehicle_economics["revenue_inr"] /
        vehicle_economics["total_attempts"]
    )

    vehicle_economics["revenue_per_completed_swap_inr"] = (
        vehicle_economics["revenue_inr"] /
        vehicle_economics["completed_swaps"]
    )

    display(vehicle_economics.round(2))


In [ ]:
# STEP 34: Vehicle-class economics — robust version

# Find vehicle class column automatically
vehicle_candidates = [
    c for c in swap_clean.columns
    if "vehicle_class" in c.lower()
]

print("Vehicle class columns found:", vehicle_candidates)

if not vehicle_candidates:
    print("\nNo vehicle class column found.")
    print("Columns containing 'vehicle':")
    print([
        c for c in swap_clean.columns
        if "vehicle" in c.lower()
    ])

else:
    vehicle_col = vehicle_candidates[0]

    # Find the amount/revenue column automatically
    amount_candidates = [
        c for c in swap_clean.columns
        if any(x in c.lower() for x in [
            "amount_charged",
            "revenue",
            "list_price"
        ])
    ]

    print("Amount columns found:", amount_candidates)

    if not amount_candidates:
        print("\nNo revenue/amount column found.")
    else:
        amount_col = (
            "amount_charged_clean"
            if "amount_charged_clean" in swap_clean.columns
            else amount_candidates[0]
        )

        vehicle_economics = (
            swap_clean
            .groupby(vehicle_col)
            .agg(
                total_attempts=("event_type", "size"),
                completed_swaps=("is_completed", "sum"),
                failed_attempts=("is_failed", "sum"),
                revenue_inr=(amount_col, "sum"),
                avg_amount_charged_inr=(amount_col, "mean")
            )
            .reset_index()
        )

        # Avoid division errors
        vehicle_economics["completion_rate_pct"] = np.where(
            vehicle_economics["total_attempts"] > 0,
            vehicle_economics["completed_swaps"] /
            vehicle_economics["total_attempts"] * 100,
            np.nan
        )

        vehicle_economics["failure_rate_pct"] = np.where(
            vehicle_economics["total_attempts"] > 0,
            vehicle_economics["failed_attempts"] /
            vehicle_economics["total_attempts"] * 100,
            np.nan
        )

        vehicle_economics["revenue_per_attempt_inr"] = np.where(
            vehicle_economics["total_attempts"] > 0,
            vehicle_economics["revenue_inr"] /
            vehicle_economics["total_attempts"],
            np.nan
        )

        vehicle_economics["revenue_per_completed_swap_inr"] = np.where(
            vehicle_economics["completed_swaps"] > 0,
            vehicle_economics["revenue_inr"] /
            vehicle_economics["completed_swaps"],
            np.nan
        )

        display(vehicle_economics.round(2))

In [ ]:
# STEP 35: Discount level vs service performance

# Check available pricing columns
pricing_cols = [
    c for c in swap_clean.columns
    if any(x in c.lower() for x in [
        "price", "discount", "amount"
    ])
]

print("Pricing-related columns found:")
print(pricing_cols)

# Create discount bands
if "discount_clean" in swap_clean.columns:

    swap_clean["discount_band"] = pd.cut(
        swap_clean["discount_clean"],
        bins=[-0.01, 0, 5, 10, 20, 100],
        labels=[
            "No discount",
            "0–5%",
            "5–10%",
            "10–20%",
            "20%+"
        ],
        include_lowest=True
    )

    pricing_service = (
        swap_clean
        .dropna(subset=["discount_band"])
        .groupby("discount_band", observed=False)
        .agg(
            total_attempts=("event_type", "size"),
            completed_swaps=("is_completed", "sum"),
            failed_attempts=("is_failed", "sum"),
            revenue_inr=("amount_charged_clean", "sum"),
            avg_amount_charged_inr=("amount_charged_clean", "mean")
        )
        .reset_index()
    )

    pricing_service["completion_rate_pct"] = (
        pricing_service["completed_swaps"] /
        pricing_service["total_attempts"] * 100
    )

    pricing_service["failure_rate_pct"] = (
        pricing_service["failed_attempts"] /
        pricing_service["total_attempts"] * 100
    )

    pricing_service["revenue_per_attempt_inr"] = (
        pricing_service["revenue_inr"] /
        pricing_service["total_attempts"]
    )

    display(pricing_service.round(2))

else:
    print("\n'discount_clean' was not found.")
    print("Please check the pricing-related columns listed above.")

In [ ]:
# STEP 36: Fleet partner economics

# Find partner column
partner_candidates = [
    c for c in swap_clean.columns
    if "partner_id" in c.lower()
]

print("Partner columns found:", partner_candidates)

if not partner_candidates:
    print("\nNo partner column found.")
    print([
        c for c in swap_clean.columns
        if "partner" in c.lower()
    ])

else:
    partner_col = partner_candidates[0]

    # Use cleaned amount if available
    if "amount_charged_clean" in swap_clean.columns:
        amount_col = "amount_charged_clean"
    else:
        amount_candidates = [
            c for c in swap_clean.columns
            if "amount_charged" in c.lower()
        ]
        amount_col = amount_candidates[0] if amount_candidates else None

    if amount_col is None:
        print("No amount-charged column found.")
    else:
        partner_economics = (
            swap_clean
            .groupby(partner_col)
            .agg(
                total_attempts=("event_type", "size"),
                completed_swaps=("is_completed", "sum"),
                failed_attempts=("is_failed", "sum"),
                abandoned_swaps=("is_abandoned", "sum"),
                revenue_inr=(amount_col, "sum")
            )
            .reset_index()
        )

        partner_economics["completion_rate_pct"] = (
            partner_economics["completed_swaps"] /
            partner_economics["total_attempts"] * 100
        )

        partner_economics["failure_rate_pct"] = (
            partner_economics["failed_attempts"] /
            partner_economics["total_attempts"] * 100
        )

        partner_economics["abandonment_rate_pct"] = (
            partner_economics["abandoned_swaps"] /
            partner_economics["total_attempts"] * 100
        )

        partner_economics["revenue_per_attempt_inr"] = (
            partner_economics["revenue_inr"] /
            partner_economics["total_attempts"]
        )

        partner_economics["revenue_per_completed_swap_inr"] = np.where(
            partner_economics["completed_swaps"] > 0,
            partner_economics["revenue_inr"] /
            partner_economics["completed_swaps"],
            np.nan
        )

        partner_economics = partner_economics.sort_values(
            "revenue_inr",
            ascending=False
        )

        display(partner_economics.round(2))

In [ ]:
# STEP 37: Rider retention analysis

# Work only with valid rider IDs
rider_events = swap_clean.dropna(
    subset=["rider_id", "event_ts"]
).copy()

# First swap date for each rider
first_swap = (
    rider_events
    .groupby("rider_id")["event_ts"]
    .min()
    .reset_index(name="first_swap_ts")
)

# Count swaps for each rider
rider_activity = (
    rider_events
    .groupby("rider_id")
    .agg(
        total_attempts=("event_type", "size"),
        completed_swaps=("is_completed", "sum"),
        last_swap_ts=("event_ts", "max")
    )
    .reset_index()
)

# Combine
rider_retention = first_swap.merge(
    rider_activity,
    on="rider_id",
    how="left"
)

# Days active in the observed dataset
rider_retention["active_days"] = (
    rider_retention["last_swap_ts"] -
    rider_retention["first_swap_ts"]
).dt.total_seconds() / (60 * 60 * 24)

# Returned after first activity
rider_retention["returned"] = (
    rider_retention["total_attempts"] > 1
)

# Retention summary
retention_summary = pd.DataFrame({
    "metric": [
        "Unique riders",
        "Riders with repeat activity",
        "Riders with one attempt only",
        "Repeat activity rate %",
        "One-attempt rate %"
    ],
    "value": [
        rider_retention["rider_id"].nunique(),
        rider_retention["returned"].sum(),
        (~rider_retention["returned"]).sum(),
        rider_retention["returned"].mean() * 100,
        (~rider_retention["returned"]).mean() * 100
    ]
})

display(retention_summary.round(2))

In [ ]:
# STEP 38: First-swap experience vs rider retention

# Sort each rider's events chronologically
rider_events_sorted = (
    swap_clean
    .dropna(subset=["rider_id", "event_ts"])
    .sort_values(["rider_id", "event_ts"])
    .copy()
)

# Take the first recorded event for each rider
first_event = (
    rider_events_sorted
    .groupby("rider_id", as_index=False)
    .first()
)

# Identify riders who returned
return_status = (
    rider_retention[
        ["rider_id", "returned"]
    ]
    .copy()
)

# Combine first experience with retention
first_experience = first_event.merge(
    return_status,
    on="rider_id",
    how="left"
)

# Analyze retention by first event type
first_event_retention = (
    first_experience
    .groupby("event_type")
    .agg(
        riders=("rider_id", "nunique"),
        returned_riders=("returned", "sum"),
        return_rate_pct=("returned", "mean")
    )
    .reset_index()
)

first_event_retention["return_rate_pct"] *= 100

# Sort by number of riders
first_event_retention = first_event_retention.sort_values(
    "riders",
    ascending=False
)

display(first_event_retention.round(2))

In [ ]:
# STEP 39: First-swap city vs rider retention

# Sort events chronologically for each rider
rider_events_sorted = (
    swap_clean
    .dropna(subset=["rider_id", "event_ts"])
    .sort_values(["rider_id", "event_ts"])
    .copy()
)

# First recorded event for each rider
first_event = (
    rider_events_sorted
    .groupby("rider_id", as_index=False)
    .first()
)

# Find the available city column
city_candidates = [
    c for c in ["station_city", "city_clean", "city"]
    if c in first_event.columns
]

if not city_candidates:
    print("No city column found.")
    print([
        c for c in first_event.columns
        if "city" in c.lower()
    ])

else:
    first_city_col = city_candidates[0]

    # Add retention status
    first_city_retention = first_event.merge(
        rider_retention[["rider_id", "returned"]],
        on="rider_id",
        how="left"
    )

    city_retention = (
        first_city_retention
        .groupby(first_city_col)
        .agg(
            riders=("rider_id", "nunique"),
            returned_riders=("returned", "sum"),
            return_rate_pct=("returned", "mean")
        )
        .reset_index()
    )

    city_retention["return_rate_pct"] *= 100

    city_retention = city_retention.sort_values(
        "riders",
        ascending=False
    )

    display(city_retention.round(2))

In [ ]:
# STEP 39: First-swap city vs rider retention

# Sort events chronologically for each rider
rider_events_sorted = (
    swap_clean
    .dropna(subset=["rider_id", "event_ts"])
    .sort_values(["rider_id", "event_ts"])
    .copy()
)

# First recorded event for each rider
first_event = (
    rider_events_sorted
    .groupby("rider_id", as_index=False)
    .first()
)

# Find the available city column
city_candidates = [
    c for c in ["station_city", "city_clean", "city"]
    if c in first_event.columns
]

if not city_candidates:
    print("No city column found.")
    print([
        c for c in first_event.columns
        if "city" in c.lower()
    ])

else:
    first_city_col = city_candidates[0]

    # Add retention status
    first_city_retention = first_event.merge(
        rider_retention[["rider_id", "returned"]],
        on="rider_id",
        how="left"
    )

    city_retention = (
        first_city_retention
        .groupby(first_city_col)
        .agg(
            riders=("rider_id", "nunique"),
            returned_riders=("returned", "sum"),
            return_rate_pct=("returned", "mean")
        )
        .reset_index()
    )

    city_retention["return_rate_pct"] *= 100

    city_retention = city_retention.sort_values(
        "riders",
        ascending=False
    )

    display(city_retention.round(2))

In [ ]:
# STEP 41: First-swap failure vs rider retention

# Sort events chronologically
rider_events_sorted = (
    swap_clean
    .dropna(subset=["rider_id", "event_ts"])
    .sort_values(["rider_id", "event_ts"])
    .copy()
)

# Get each rider's first recorded event
first_event = (
    rider_events_sorted
    .groupby("rider_id", as_index=False)
    .first()
)

# Add retention status
first_failure_retention = first_event.merge(
    rider_retention[["rider_id", "returned"]],
    on="rider_id",
    how="left"
)

# Classify first experience
first_failure_retention["first_experience"] = np.select(
    [
        first_failure_retention["is_completed"],
        first_failure_retention["is_failed"],
        first_failure_retention["is_abandoned"]
    ],
    [
        "Completed",
        "Failed",
        "Abandoned"
    ],
    default="Other"
)

# Retention by first experience
failure_retention = (
    first_failure_retention
    .groupby("first_experience")
    .agg(
        riders=("rider_id", "nunique"),
        returned_riders=("returned", "sum"),
        return_rate_pct=("returned", "mean")
    )
    .reset_index()
)

failure_retention["return_rate_pct"] *= 100

display(
    failure_retention.round(2)
)

In [ ]:
# STEP 42: Build rider-level retention analysis dataset

# Start with rider retention
retention_analysis = rider_retention.copy()

# Get each rider's first recorded event
first_events = (
    swap_clean
    .dropna(subset=["rider_id", "event_ts"])
    .sort_values(["rider_id", "event_ts"])
    .groupby("rider_id", as_index=False)
    .first()
)

# Keep useful first-experience fields that actually exist
possible_cols = [
    "rider_id",
    "event_ts",
    "event_type",
    "station_city",
    "vehicle_class",
    "queue_wait_min_clean",
    "battery_in_current_soh_pct",
    "battery_in_supplier",
    "charger_generation",
    "amount_charged_clean",
    "discount_clean"
]

available_cols = [
    c for c in possible_cols
    if c in first_events.columns
]

first_experience = first_events[available_cols].copy()

# Rename first-event columns
rename_map = {
    "event_ts": "first_event_ts",
    "event_type": "first_event_type",
    "station_city": "first_station_city",
    "vehicle_class": "first_vehicle_class",
    "queue_wait_min_clean": "first_queue_wait_min",
    "battery_in_current_soh_pct": "first_battery_soh_pct",
    "battery_in_supplier": "first_battery_supplier",
    "charger_generation": "first_charger_generation",
    "amount_charged_clean": "first_amount_charged_inr",
    "discount_clean": "first_discount_pct"
}

first_experience = first_experience.rename(
    columns={
        k: v for k, v in rename_map.items()
        if k in first_experience.columns
    }
)

# Merge with retention data
retention_analysis = retention_analysis.merge(
    first_experience,
    on="rider_id",
    how="left"
)

# Create simple first-experience classification
if "first_event_type" in retention_analysis.columns:
    retention_analysis["first_experience_group"] = np.select(
        [
            retention_analysis["first_event_type"] == "swap_completed",
            retention_analysis["first_event_type"].isin([
                "failed_no_charged_battery",
                "failed_system_error"
            ]),
            retention_analysis["first_event_type"] == "abandoned_queue"
        ],
        [
            "Completed",
            "Failed",
            "Abandoned"
        ],
        default="Other"
    )

print("Retention analysis dataset shape:", retention_analysis.shape)

display(
    retention_analysis.head()
)

In [ ]:
# STEP 43: Returning vs non-returning rider comparison

# Create two groups
returning = retention_analysis[
    retention_analysis["returned"] == True
]

non_returning = retention_analysis[
    retention_analysis["returned"] == False
]

print("RETURNING RIDERS:", len(returning))
print("NON-RETURNING RIDERS:", len(non_returning))

# Build comparison table
comparison = []

# Queue wait
if "first_queue_wait_min" in retention_analysis.columns:
    comparison.append({
        "factor": "First queue wait (min)",
        "returning_avg": returning["first_queue_wait_min"].mean(),
        "non_returning_avg": non_returning["first_queue_wait_min"].mean()
    })

# Battery SOH
if "first_battery_soh_pct" in retention_analysis.columns:
    comparison.append({
        "factor": "First battery SOH (%)",
        "returning_avg": returning["first_battery_soh_pct"].mean(),
        "non_returning_avg": non_returning["first_battery_soh_pct"].mean()
    })

# Amount charged
if "first_amount_charged_inr" in retention_analysis.columns:
    comparison.append({
        "factor": "First amount charged (INR)",
        "returning_avg": returning["first_amount_charged_inr"].mean(),
        "non_returning_avg": non_returning["first_amount_charged_inr"].mean()
    })

# Discount
if "first_discount_pct" in retention_analysis.columns:
    comparison.append({
        "factor": "First discount (%)",
        "returning_avg": returning["first_discount_pct"].mean(),
        "non_returning_avg": non_returning["first_discount_pct"].mean()
    })

comparison_df = pd.DataFrame(comparison)

display(comparison_df.round(2))

In [ ]:
# STEP 44: Retention rate by first experience

if "first_experience_group" not in retention_analysis.columns:
    print("first_experience_group not found.")
else:
    retention_by_experience = (
        retention_analysis
        .groupby("first_experience_group")
        .agg(
            riders=("rider_id", "nunique"),
            returned_riders=("returned", "sum")
        )
        .reset_index()
    )

    retention_by_experience["return_rate_pct"] = (
        retention_by_experience["returned_riders"] /
        retention_by_experience["riders"] * 100
    )

    retention_by_experience["non_return_rate_pct"] = (
        100 - retention_by_experience["return_rate_pct"]
    )

    display(
        retention_by_experience.round(2)
    )

In [ ]:
# STEP 45: Root-cause factor summary

root_cause_rows = []

# 1. First experience
if "first_experience_group" in retention_analysis.columns:
    temp = (
        retention_analysis
        .groupby("first_experience_group")
        .agg(
            riders=("rider_id", "nunique"),
            return_rate_pct=("returned", "mean")
        )
        .reset_index()
    )

    temp["return_rate_pct"] *= 100

    for _, row in temp.iterrows():
        root_cause_rows.append({
            "factor": "First experience",
            "segment": row["first_experience_group"],
            "sample_size": row["riders"],
            "return_rate_pct": row["return_rate_pct"]
        })

# 2. First queue experience
if "first_queue_wait_min" in retention_analysis.columns:
    temp = retention_analysis.copy()

    temp["queue_band"] = pd.cut(
        temp["first_queue_wait_min"],
        bins=[-0.01, 2, 5, 10, 20, np.inf],
        labels=[
            "0–2 min",
            "2–5 min",
            "5–10 min",
            "10–20 min",
            "20+ min"
        ]
    )

    temp = (
        temp.dropna(subset=["queue_band"])
        .groupby("queue_band", observed=False)
        .agg(
            riders=("rider_id", "nunique"),
            return_rate_pct=("returned", "mean")
        )
        .reset_index()
    )

    temp["return_rate_pct"] *= 100

    for _, row in temp.iterrows():
        root_cause_rows.append({
            "factor": "First queue wait",
            "segment": row["queue_band"],
            "sample_size": row["riders"],
            "return_rate_pct": row["return_rate_pct"]
        })

# 3. First battery SOH
if "first_battery_soh_pct" in retention_analysis.columns:
    temp = retention_analysis.copy()

    temp["soh_band"] = pd.cut(
        temp["first_battery_soh_pct"],
        bins=[0, 60, 70, 80, 90, 100],
        labels=[
            "<60%",
            "60–70%",
            "70–80%",
            "80–90%",
            "90–100%"
        ]
    )

    temp = (
        temp.dropna(subset=["soh_band"])
        .groupby("soh_band", observed=False)
        .agg(
            riders=("rider_id", "nunique"),
            return_rate_pct=("returned", "mean")
        )
        .reset_index()
    )

    temp["return_rate_pct"] *= 100

    for _, row in temp.iterrows():
        root_cause_rows.append({
            "factor": "First battery SOH",
            "segment": row["soh_band"],
            "sample_size": row["riders"],
            "return_rate_pct": row["return_rate_pct"]
        })

root_cause_summary = pd.DataFrame(root_cause_rows)

display(root_cause_summary.round(2))

In [ ]:
# STEP 46: Retention root-cause visualization

if "retention_by_experience" not in globals():
    print("Run Step 44 first.")
else:
    plt.figure(figsize=(10, 6))

    plot_data = retention_by_experience.sort_values(
        "return_rate_pct",
        ascending=False
    )

    sns.barplot(
        data=plot_data,
        x="first_experience_group",
        y="return_rate_pct"
    )

    plt.title("Rider Return Rate by First Service Experience")
    plt.xlabel("First Recorded Experience")
    plt.ylabel("Return Rate (%)")
    plt.ylim(0, 100)

    # Add percentage labels
    for i, value in enumerate(plot_data["return_rate_pct"]):
        plt.text(
            i,
            value + 1,
            f"{value:.1f}%",
            ha="center"
        )

    plt.tight_layout()
    plt.show()

In [ ]:
# STEP 47: Station hotspots with operational characteristics

# Select station attributes that actually exist
station_attributes = [
    "station_id",
    "city_clean",
    "city",
    "zone",
    "location_type",
    "host_type",
    "expansion_wave",
    "charger_generation",
    "connectivity_tier"
]

available_attributes = [
    c for c in station_attributes
    if c in stations.columns
]

# Keep one row per station
station_info = (
    stations[available_attributes]
    .drop_duplicates(subset=["station_id"])
)

# Merge with station performance
station_hotspots = station_service_filtered.merge(
    station_info,
    on="station_id",
    how="left"
)

# Show highest-failure stations with their characteristics
station_hotspots = station_hotspots.sort_values(
    "failure_rate_pct",
    ascending=False
)

display(
    station_hotspots.head(15).round(2)
)

In [ ]:
# STEP 48: Network performance trends

fig, axes = plt.subplots(3, 1, figsize=(12, 12))

# 1. Completed swaps
axes[0].plot(
    monthly_kpi["event_month"].astype(str),
    monthly_kpi["completed_swaps"],
    marker="o"
)
axes[0].set_title("Completed Swaps Over Time")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Completed Swaps")
axes[0].tick_params(axis="x", rotation=45)

# 2. Failure rate
axes[1].plot(
    monthly_kpi["event_month"].astype(str),
    monthly_kpi["failure_rate_pct"],
    marker="o"
)
axes[1].set_title("Failure Rate Over Time")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Failure Rate (%)")
axes[1].tick_params(axis="x", rotation=45)

# 3. Revenue
axes[2].plot(
    monthly_kpi["event_month"].astype(str),
    monthly_kpi["revenue_inr"],
    marker="o"
)
axes[2].set_title("Revenue Over Time")
axes[2].set_xlabel("Month")
axes[2].set_ylabel("Revenue (INR)")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# STEP 49: Monthly revenue and pricing performance

monthly_pricing = (
    swap_clean
    .groupby("event_month")
    .agg(
        total_attempts=("event_type", "size"),
        completed_swaps=("is_completed", "sum"),
        revenue_inr=("amount_charged_clean", "sum"),
        avg_amount_charged_inr=("amount_charged_clean", "mean"),
        avg_discount_pct=("discount_clean", "mean")
    )
    .reset_index()
)

monthly_pricing["revenue_per_attempt_inr"] = (
    monthly_pricing["revenue_inr"] /
    monthly_pricing["total_attempts"]
)

monthly_pricing["revenue_per_completed_swap_inr"] = np.where(
    monthly_pricing["completed_swaps"] > 0,
    monthly_pricing["revenue_inr"] /
    monthly_pricing["completed_swaps"],
    np.nan
)

display(monthly_pricing.round(2))

In [ ]:
# STEP 49: Monthly revenue and pricing performance — fixed

# Check available pricing columns
print("Pricing columns available:")

pricing_candidates = [
    c for c in swap_clean.columns
    if any(word in c.lower() for word in [
        "amount", "price", "discount", "revenue"
    ])
]

print(pricing_candidates)

# Select amount column
if "amount_charged_clean" in swap_clean.columns:
    amount_col = "amount_charged_clean"
elif "amount_charged" in swap_clean.columns:
    amount_col = "amount_charged"
else:
    amount_options = [
        c for c in pricing_candidates
        if "amount_charged" in c.lower()
    ]

    if amount_options:
        amount_col = amount_options[0]
    else:
        amount_col = None

# Select discount column
if "discount_clean" in swap_clean.columns:
    discount_col = "discount_clean"
elif "discount_pct" in swap_clean.columns:
    discount_col = "discount_pct"
else:
    discount_options = [
        c for c in pricing_candidates
        if "discount" in c.lower()
    ]

    discount_col = discount_options[0] if discount_options else None

print("\nAmount column:", amount_col)
print("Discount column:", discount_col)

# Stop if amount is unavailable
if amount_col is None:
    print("\nNo amount/revenue column was found.")
else:

    # Basic monthly aggregation
    monthly_pricing = (
        swap_clean
        .groupby("event_month")
        .agg(
            total_attempts=("event_type", "size"),
            completed_swaps=("is_completed", "sum"),
            revenue_inr=(amount_col, "sum"),
            avg_amount_charged_inr=(amount_col, "mean")
        )
        .reset_index()
    )

    # Add discount only if available
    if discount_col is not None:
        discount_data = (
            swap_clean
            .groupby("event_month")[discount_col]
            .mean()
            .reset_index(name="avg_discount_pct")
        )

        monthly_pricing = monthly_pricing.merge(
            discount_data,
            on="event_month",
            how="left"
        )

    # Revenue per attempt
    monthly_pricing["revenue_per_attempt_inr"] = np.where(
        monthly_pricing["total_attempts"] > 0,
        monthly_pricing["revenue_inr"] /
        monthly_pricing["total_attempts"],
        np.nan
    )

    # Revenue per completed swap
    monthly_pricing["revenue_per_completed_swap_inr"] = np.where(
        monthly_pricing["completed_swaps"] > 0,
        monthly_pricing["revenue_inr"] /
        monthly_pricing["completed_swaps"],
        np.nan
    )

    monthly_pricing = monthly_pricing.sort_values(
        "event_month"
    )

    display(monthly_pricing.round(2))

In [ ]:
# STEP 50: Identify available cost and margin inputs

cost_keywords = [
    "cost",
    "tariff",
    "rent",
    "maintenance",
    "energy",
    "capacity",
    "purchase"
]

cost_columns = [
    c for c in swap_clean.columns
    if any(k in c.lower() for k in cost_keywords)
]

print("Cost / margin related columns found:")
for c in cost_columns:
    print(" -", c)

print("\nSample of available cost-related data:")
display(swap_clean[cost_columns].head())

print("\nData types:")
display(swap_clean[cost_columns].dtypes)

In [ ]:
# STEP 51: Support ticket analysis

# Standardize ticket city if available
ticket_city_col = next(
    (c for c in support_tickets.columns if c.lower() in ["city", "home_city"]),
    None
)

ticket_category_col = next(
    (c for c in support_tickets.columns
     if "category" in c.lower()),
    None
)

print("City column:", ticket_city_col)
print("Category column:", ticket_category_col)

if ticket_category_col is None:
    print("No ticket category column found.")
else:

    # Overall complaint categories
    ticket_summary = (
        support_tickets[ticket_category_col]
        .value_counts(dropna=False)
        .reset_index()
    )

    ticket_summary.columns = [
        "ticket_category",
        "ticket_count"
    ]

    ticket_summary["share_pct"] = (
        ticket_summary["ticket_count"] /
        ticket_summary["ticket_count"].sum() * 100
    )

    print("Overall support ticket categories:")
    display(ticket_summary.round(2))

    # City-level ticket analysis
    if ticket_city_col is not None:

        ticket_city = (
            support_tickets
            .groupby([ticket_city_col, ticket_category_col])
            .size()
            .reset_index(name="ticket_count")
        )

        ticket_city["share_within_city_pct"] = (
            ticket_city["ticket_count"] /
            ticket_city.groupby(ticket_city_col)["ticket_count"]
            .transform("sum") * 100
        )

        print("\nTop complaint categories by city:")
        display(
            ticket_city
            .sort_values(
                [ticket_city_col, "ticket_count"],
                ascending=[True, False]
            )
            .groupby(ticket_city_col)
            .head(5)
            .round(2)
        )

    # Plot overall complaint categories
    plot_data = ticket_summary.head(10).sort_values(
        "ticket_count"
    )

    plt.figure(figsize=(10, 6))
    plt.barh(
        plot_data["ticket_category"].astype(str),
        plot_data["ticket_count"]
    )
    plt.xlabel("Number of Support Tickets")
    plt.ylabel("Complaint Category")
    plt.title("Top Customer Support Complaint Categories")
    plt.tight_layout()
    plt.show()

In [ ]:
# STEP 52: Expansion wave performance

wave_col = "expansion_wave"

if wave_col not in swap_clean.columns:
    print("Expansion wave column not found.")
else:

    expansion_performance = (
        swap_clean
        .groupby(wave_col)
        .agg(
            total_attempts=("event_type", "size"),
            completed_swaps=("is_completed", "sum"),
            failed_attempts=("is_failed", "sum"),
            abandoned_swaps=("is_abandoned", "sum"),
            revenue_inr=("amount_charged_clean", "sum")
        )
        .reset_index()
    )

    expansion_performance["completion_rate_pct"] = (
        expansion_performance["completed_swaps"] /
        expansion_performance["total_attempts"] * 100
    )

    expansion_performance["failure_rate_pct"] = (
        expansion_performance["failed_attempts"] /
        expansion_performance["total_attempts"] * 100
    )

    expansion_performance["abandonment_rate_pct"] = (
        expansion_performance["abandoned_swaps"] /
        expansion_performance["total_attempts"] * 100
    )

    expansion_performance["revenue_per_attempt_inr"] = (
        expansion_performance["revenue_inr"] /
        expansion_performance["total_attempts"]
    )

    expansion_performance["revenue_per_completed_swap_inr"] = np.where(
        expansion_performance["completed_swaps"] > 0,
        expansion_performance["revenue_inr"] /
        expansion_performance["completed_swaps"],
        np.nan
    )

    display(
        expansion_performance
        .sort_values(wave_col)
        .round(2)
    )

    # Visual comparison
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].bar(
        expansion_performance[wave_col].astype(str),
        expansion_performance["failure_rate_pct"]
    )
    axes[0].set_title("Failure Rate by Expansion Wave")
    axes[0].set_ylabel("Failure Rate (%)")
    axes[0].set_xlabel("Expansion Wave")

    axes[1].bar(
        expansion_performance[wave_col].astype(str),
        expansion_performance["completion_rate_pct"]
    )
    axes[1].set_title("Completion Rate by Expansion Wave")
    axes[1].set_ylabel("Completion Rate (%)")
    axes[1].set_xlabel("Expansion Wave")

    plt.tight_layout()
    plt.show()

In [ ]:
# STEP 52: Expansion Wave Performance — FIXED

# Find expansion wave column
wave_col = next(
    (c for c in swap_clean.columns
     if c.lower() == "expansion_wave"),
    None
)

# Find revenue/amount column
amount_col = next(
    (c for c in ["amount_charged_clean", "amount_charged", "revenue_inr"]
     if c in swap_clean.columns),
    None
)

print("Expansion wave column:", wave_col)
print("Revenue column:", amount_col)

if wave_col is None:
    print("\n❌ Expansion wave column was not found.")
    print("Available station-related columns:")
    print([
        c for c in swap_clean.columns
        if any(x in c.lower() for x in ["station", "wave", "charger"])
    ])

else:

    # Core performance
    expansion_performance = (
        swap_clean
        .groupby(wave_col)
        .agg(
            total_attempts=("event_type", "size"),
            completed_swaps=("is_completed", "sum"),
            failed_attempts=("is_failed", "sum"),
            abandoned_swaps=("is_abandoned", "sum")
        )
        .reset_index()
    )

    # Revenue if available
    if amount_col is not None:
        revenue = (
            swap_clean
            .groupby(wave_col)[amount_col]
            .sum()
            .reset_index(name="revenue_inr")
        )

        expansion_performance = expansion_performance.merge(
            revenue,
            on=wave_col,
            how="left"
        )

    # Rates
    expansion_performance["completion_rate_pct"] = (
        expansion_performance["completed_swaps"] /
        expansion_performance["total_attempts"] * 100
    )

    expansion_performance["failure_rate_pct"] = (
        expansion_performance["failed_attempts"] /
        expansion_performance["total_attempts"] * 100
    )

    expansion_performance["abandonment_rate_pct"] = (
        expansion_performance["abandoned_swaps"] /
        expansion_performance["total_attempts"] * 100
    )

    # Revenue metrics
    if "revenue_inr" in expansion_performance.columns:

        expansion_performance["revenue_per_attempt_inr"] = (
            expansion_performance["revenue_inr"] /
            expansion_performance["total_attempts"]
        )

        expansion_performance["revenue_per_completed_swap_inr"] = np.where(
            expansion_performance["completed_swaps"] > 0,
            expansion_performance["revenue_inr"] /
            expansion_performance["completed_swaps"],
            np.nan
        )

    # Display
    display(
        expansion_performance
        .sort_values(wave_col)
        .round(2)
    )

    # Simple charts
    plt.figure(figsize=(9, 5))

    plt.bar(
        expansion_performance[wave_col].astype(str),
        expansion_performance["failure_rate_pct"]
    )

    plt.title("Failure Rate by Expansion Wave")
    plt.xlabel("Expansion Wave")
    plt.ylabel("Failure Rate (%)")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(9, 5))

    plt.bar(
        expansion_performance[wave_col].astype(str),
        expansion_performance["completion_rate_pct"]
    )

    plt.title("Completion Rate by Expansion Wave")
    plt.xlabel("Expansion Wave")
    plt.ylabel("Completion Rate (%)")
    plt.tight_layout()
    plt.show()

In [ ]:
# STEP 53: Cross-factor retention comparison

comparison_results = []

# 1. First experience
if "first_experience_group" in retention_analysis.columns:
    temp = (
        retention_analysis
        .groupby("first_experience_group")
        .agg(
            riders=("rider_id", "size"),
            return_rate_pct=("returned", "mean")
        )
        .reset_index()
    )

    temp["return_rate_pct"] *= 100
    temp["factor"] = "First Experience"
    temp["segment"] = temp["first_experience_group"].astype(str)

    comparison_results.append(
        temp[["factor", "segment", "riders", "return_rate_pct"]]
    )

# 2. First queue wait
if "first_queue_wait_min" in retention_analysis.columns:

    temp_data = retention_analysis.copy()

    temp_data["queue_band"] = pd.cut(
        temp_data["first_queue_wait_min"],
        bins=[-np.inf, 2, 5, 10, 20, np.inf],
        labels=["0–2 min", "2–5 min", "5–10 min",
                "10–20 min", "20+ min"]
    )

    temp = (
        temp_data
        .groupby("queue_band", observed=False)
        .agg(
            riders=("rider_id", "size"),
            return_rate_pct=("returned", "mean")
        )
        .reset_index()
    )

    temp["return_rate_pct"] *= 100
    temp["factor"] = "First Queue Wait"
    temp["segment"] = temp["queue_band"].astype(str)

    comparison_results.append(
        temp[["factor", "segment", "riders", "return_rate_pct"]]
    )

# 3. First battery SOH
if "first_battery_soh_pct" in retention_analysis.columns:

    temp_data = retention_analysis.copy()

    temp_data["soh_band"] = pd.cut(
        temp_data["first_battery_soh_pct"],
        bins=[-np.inf, 60, 70, 80, 90, 100],
        labels=["<60%", "60–70%", "70–80%",
                "80–90%", "90–100%"]
    )

    temp = (
        temp_data
        .groupby("soh_band", observed=False)
        .agg(
            riders=("rider_id", "size"),
            return_rate_pct=("returned", "mean")
        )
        .reset_index()
    )

    temp["return_rate_pct"] *= 100
    temp["factor"] = "First Battery SOH"
    temp["segment"] = temp["soh_band"].astype(str)

    comparison_results.append(
        temp[["factor", "segment", "riders", "return_rate_pct"]]
    )

# Combine results
if comparison_results:

    root_cause_comparison = pd.concat(
        comparison_results,
        ignore_index=True
    )

    root_cause_comparison = (
        root_cause_comparison
        .sort_values(
            ["factor", "return_rate_pct"],
            ascending=[True, False]
        )
    )

    display(
        root_cause_comparison.round(2)
    )

else:
    print("No retention comparison variables were available.")

In [ ]:
# STEP 54: Final Network KPI Summary

# Create overall network KPIs
total_attempts = len(swap_clean)

completed_swaps = int(swap_clean["is_completed"].sum())
failed_attempts = int(swap_clean["is_failed"].sum())
abandoned_swaps = int(swap_clean["is_abandoned"].sum())

completion_rate = completed_swaps / total_attempts * 100
failure_rate = failed_attempts / total_attempts * 100
abandonment_rate = abandoned_swaps / total_attempts * 100

# Revenue
if "amount_charged_clean" in swap_clean.columns:
    total_revenue = swap_clean["amount_charged_clean"].sum()
else:
    total_revenue = np.nan

revenue_per_attempt = (
    total_revenue / total_attempts
    if total_attempts > 0 else np.nan
)

revenue_per_completed = (
    total_revenue / completed_swaps
    if completed_swaps > 0 else np.nan
)

# Cities and stations
if "station_id" in swap_clean.columns:
    active_stations = swap_clean["station_id"].nunique()
else:
    active_stations = np.nan

if "station_city" in swap_clean.columns:
    active_cities = swap_clean["station_city"].nunique()
elif "city_clean" in swap_clean.columns:
    active_cities = swap_clean["city_clean"].nunique()
else:
    active_cities = np.nan

# Riders
if "rider_id" in swap_clean.columns:
    unique_riders = swap_clean["rider_id"].nunique()
else:
    unique_riders = np.nan

# Date coverage
start_date = swap_clean["event_ts"].min()
end_date = swap_clean["event_ts"].max()

final_kpis = pd.DataFrame({
    "Metric": [
        "Analysis Start Date",
        "Analysis End Date",
        "Total Swap Attempts",
        "Completed Swaps",
        "Failed Attempts",
        "Abandoned Swaps",
        "Completion Rate (%)",
        "Failure Rate (%)",
        "Abandonment Rate (%)",
        "Total Revenue (INR)",
        "Revenue per Attempt (INR)",
        "Revenue per Completed Swap (INR)",
        "Unique Riders",
        "Active Stations",
        "Active Cities"
    ],
    "Value": [
        start_date,
        end_date,
        total_attempts,
        completed_swaps,
        failed_attempts,
        abandoned_swaps,
        completion_rate,
        failure_rate,
        abandonment_rate,
        total_revenue,
        revenue_per_attempt,
        revenue_per_completed,
        unique_riders,
        active_stations,
        active_cities
    ]
})

display(final_kpis.round(2))

In [ ]:
# STEP 55: Final Competition Dashboard

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ---------------------------------------------------------
# 1. Monthly completed swaps
# ---------------------------------------------------------
if "monthly_kpi" in globals():

    axes[0, 0].plot(
        monthly_kpi["event_month"].astype(str),
        monthly_kpi["completed_swaps"],
        marker="o"
    )

    axes[0, 0].set_title("Completed Swaps Over Time")
    axes[0, 0].set_xlabel("Month")
    axes[0, 0].set_ylabel("Completed Swaps")
    axes[0, 0].tick_params(axis="x", rotation=45)


# ---------------------------------------------------------
# 2. Monthly failure rate
# ---------------------------------------------------------
if "monthly_kpi" in globals():

    axes[0, 1].plot(
        monthly_kpi["event_month"].astype(str),
        monthly_kpi["failure_rate_pct"],
        marker="o"
    )

    axes[0, 1].set_title("Failure Rate Over Time")
    axes[0, 1].set_xlabel("Month")
    axes[0, 1].set_ylabel("Failure Rate (%)")
    axes[0, 1].tick_params(axis="x", rotation=45)


# ---------------------------------------------------------
# 3. City failure rate
# ---------------------------------------------------------
if "city_service" in globals():

    city_plot = (
        city_service
        .sort_values("failure_rate_pct", ascending=False)
        .head(10)
    )

    axes[1, 0].bar(
        city_plot["city"].astype(str),
        city_plot["failure_rate_pct"]
    )

    axes[1, 0].set_title("Failure Rate by City")
    axes[1, 0].set_xlabel("City")
    axes[1, 0].set_ylabel("Failure Rate (%)")
    axes[1, 0].tick_params(axis="x", rotation=45)


# ---------------------------------------------------------
# 4. Retention by first experience
# ---------------------------------------------------------
if "retention_by_experience" in globals():

    retention_plot = (
        retention_by_experience
        .sort_values("return_rate_pct", ascending=False)
    )

    axes[1, 1].bar(
        retention_plot["first_experience_group"].astype(str),
        retention_plot["return_rate_pct"]
    )

    axes[1, 1].set_title("Rider Return Rate by First Experience")
    axes[1, 1].set_xlabel("First Experience")
    axes[1, 1].set_ylabel("Return Rate (%)")
    axes[1, 1].tick_params(axis="x", rotation=45)


plt.suptitle(
    "VoltRelay Energy — Network Performance Dashboard",
    fontsize=16
)

plt.tight_layout()
plt.show()

In [ ]:
# STEP 55: Final Competition Dashboard — FIXED

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# =========================================================
# 1. Completed swaps over time
# =========================================================

if (
    "monthly_kpi" in globals()
    and "event_month" in monthly_kpi.columns
    and "completed_swaps" in monthly_kpi.columns
):

    axes[0, 0].plot(
        monthly_kpi["event_month"].astype(str),
        monthly_kpi["completed_swaps"],
        marker="o"
    )

    axes[0, 0].set_title("Completed Swaps Over Time")
    axes[0, 0].set_xlabel("Month")
    axes[0, 0].set_ylabel("Completed Swaps")
    axes[0, 0].tick_params(axis="x", rotation=45)

else:
    axes[0, 0].text(
        0.5, 0.5,
        "Monthly KPI data unavailable",
        ha="center",
        va="center"
    )


# =========================================================
# 2. Failure rate over time
# =========================================================

if (
    "monthly_kpi" in globals()
    and "event_month" in monthly_kpi.columns
    and "failure_rate_pct" in monthly_kpi.columns
):

    axes[0, 1].plot(
        monthly_kpi["event_month"].astype(str),
        monthly_kpi["failure_rate_pct"],
        marker="o"
    )

    axes[0, 1].set_title("Failure Rate Over Time")
    axes[0, 1].set_xlabel("Month")
    axes[0, 1].set_ylabel("Failure Rate (%)")
    axes[0, 1].tick_params(axis="x", rotation=45)

else:
    axes[0, 1].text(
        0.5, 0.5,
        "Failure-rate data unavailable",
        ha="center",
        va="center"
    )


# =========================================================
# 3. City failure rate
# =========================================================

if "city_service" in globals():

    city_data = city_service.copy()

    # Find city column
    city_col = next(
        (
            c for c in city_data.columns
            if c.lower() in ["city", "city_clean", "station_city"]
        ),
        None
    )

    # Find failure-rate column
    failure_col = next(
        (
            c for c in city_data.columns
            if c.lower() == "failure_rate_pct"
        ),
        None
    )

    if city_col is not None and failure_col is not None:

        city_plot = (
            city_data
            .sort_values(failure_col, ascending=False)
            .head(10)
        )

        axes[1, 0].bar(
            city_plot[city_col].astype(str),
            city_plot[failure_col]
        )

        axes[1, 0].set_title("Failure Rate by City")
        axes[1, 0].set_xlabel("City")
        axes[1, 0].set_ylabel("Failure Rate (%)")
        axes[1, 0].tick_params(axis="x", rotation=45)

    else:
        axes[1, 0].text(
            0.5, 0.5,
            "City failure data unavailable",
            ha="center",
            va="center"
        )

else:
    axes[1, 0].text(
        0.5, 0.5,
        "City service table unavailable",
        ha="center",
        va="center"
    )


# =========================================================
# 4. Rider retention
# =========================================================

if "retention_by_experience" in globals():

    retention_data = retention_by_experience.copy()

    # Find experience column
    experience_col = next(
        (
            c for c in retention_data.columns
            if "experience" in c.lower()
        ),
        None
    )

    # Find return-rate column
    return_col = next(
        (
            c for c in retention_data.columns
            if "return_rate" in c.lower()
        ),
        None
    )

    if experience_col is not None and return_col is not None:

        retention_plot = retention_data.sort_values(
            return_col,
            ascending=False
        )

        axes[1, 1].bar(
            retention_plot[experience_col].astype(str),
            retention_plot[return_col]
        )

        axes[1, 1].set_title(
            "Rider Return Rate by First Experience"
        )
        axes[1, 1].set_xlabel("First Experience")
        axes[1, 1].set_ylabel("Return Rate (%)")
        axes[1, 1].tick_params(axis="x", rotation=45)

    else:
        axes[1, 1].text(
            0.5, 0.5,
            "Retention data unavailable",
            ha="center",
            va="center"
        )

else:
    axes[1, 1].text(
        0.5, 0.5,
        "Retention table unavailable",
        ha="center",
        va="center"
    )


# =========================================================
# Final formatting
# =========================================================

plt.suptitle(
    "VoltRelay Energy — Network Performance Dashboard",
    fontsize=16
)

plt.tight_layout()
plt.show()

In [ ]:
# STEP 56: Competition Findings Summary

findings = []

# ---------------------------------------------------------
# 1. Monthly performance trend
# ---------------------------------------------------------
if "monthly_kpi" in globals() and len(monthly_kpi) >= 2:

    first = monthly_kpi.iloc[0]
    last = monthly_kpi.iloc[-1]

    findings.append({
        "Area": "Network Performance",
        "Metric": "Completed Swaps",
        "Start Value": first["completed_swaps"],
        "End Value": last["completed_swaps"],
        "Change (%)": (
            (last["completed_swaps"] / first["completed_swaps"] - 1) * 100
            if first["completed_swaps"] != 0 else np.nan
        )
    })

    findings.append({
        "Area": "Network Performance",
        "Metric": "Failure Rate",
        "Start Value": first["failure_rate_pct"],
        "End Value": last["failure_rate_pct"],
        "Change (%)": (
            last["failure_rate_pct"] - first["failure_rate_pct"]
        )
    })


# ---------------------------------------------------------
# 2. City performance
# ---------------------------------------------------------
if "city_service" in globals():

    city_data = city_service.copy()

    city_col = next(
        (
            c for c in city_data.columns
            if c.lower() in ["city", "city_clean", "station_city"]
        ),
        None
    )

    failure_col = next(
        (
            c for c in city_data.columns
            if c.lower() == "failure_rate_pct"
        ),
        None
    )

    if city_col and failure_col:

        highest = city_data.loc[
            city_data[failure_col].idxmax()
        ]

        lowest = city_data.loc[
            city_data[failure_col].idxmin()
        ]

        findings.append({
            "Area": "City Performance",
            "Metric": "Highest Failure Rate",
            "Start Value": highest[city_col],
            "End Value": highest[failure_col],
            "Change (%)": np.nan
        })

        findings.append({
            "Area": "City Performance",
            "Metric": "Lowest Failure Rate",
            "Start Value": lowest[city_col],
            "End Value": lowest[failure_col],
            "Change (%)": np.nan
        })


# ---------------------------------------------------------
# 3. Station hotspots
# ---------------------------------------------------------
if "station_hotspots" in globals() and len(station_hotspots) > 0:

    station_col = next(
        (
            c for c in station_hotspots.columns
            if c.lower() == "station_id"
        ),
        None
    )

    failure_col = next(
        (
            c for c in station_hotspots.columns
            if c.lower() == "failure_rate_pct"
        ),
        None
    )

    if station_col and failure_col:

        top_station = station_hotspots.iloc[0]

        findings.append({
            "Area": "Station Performance",
            "Metric": "Highest-Risk Station",
            "Start Value": top_station[station_col],
            "End Value": top_station[failure_col],
            "Change (%)": np.nan
        })


# ---------------------------------------------------------
# 4. Retention
# ---------------------------------------------------------
if "retention_by_experience" in globals():

    retention_data = retention_by_experience.copy()

    experience_col = next(
        (
            c for c in retention_data.columns
            if "experience" in c.lower()
        ),
        None
    )

    return_col = next(
        (
            c for c in retention_data.columns
            if "return_rate" in c.lower()
        ),
        None
    )

    if experience_col and return_col:

        highest_return = retention_data.loc[
            retention_data[return_col].idxmax()
        ]

        lowest_return = retention_data.loc[
            retention_data[return_col].idxmin()
        ]

        findings.append({
            "Area": "Rider Retention",
            "Metric": "Highest Return Segment",
            "Start Value": highest_return[experience_col],
            "End Value": highest_return[return_col],
            "Change (%)": np.nan
        })

        findings.append({
            "Area": "Rider Retention",
            "Metric": "Lowest Return Segment",
            "Start Value": lowest_return[experience_col],
            "End Value": lowest_return[return_col],
            "Change (%)": np.nan
        })


# ---------------------------------------------------------
# Display final findings table
# ---------------------------------------------------------

if findings:

    findings_summary = pd.DataFrame(findings)

    display(
        findings_summary.round(2)
    )

else:
    print("No findings could be extracted from the available analysis tables.")

No findings could be extracted from the available analysis tables.


In [ ]:
# STEP 57: Executive KPI Snapshot

# Make sure final_kpis exists
if "final_kpis" in globals():

    executive_kpis = final_kpis.copy()

    # Convert dates to readable format
    if "Value" in executive_kpis.columns:
        executive_kpis["Value"] = executive_kpis["Value"].apply(
            lambda x: x.strftime("%Y-%m-%d")
            if hasattr(x, "strftime")
            else x
        )

    display(executive_kpis)

else:
    print("final_kpis was not found. Please run STEP 54 first.")

final_kpis was not found. Please run STEP 54 first.
